# Notebook 01.1 — Viewing Zenith Angle Effects on VIIRS Nighttime Lights

## Purpose

This standalone notebook tests whether VIIRS viewing zenith angle (VZA; `Sensor_Zenith`) materially changes the inferred Haiyan nighttime lights trajectory. It evaluates:

1. spatial and temporal viewing geometry over Samar–Leyte;
2. VZA distributions and observability across GHSL settlement classes;
3. residual VZA sensitivity in direct DNB-BRDF and paired gap-filled radiance;
4. the repeating 16-day orbital pattern;
5. stability under alternative VZA thresholds; and
6. an optional nadir-equivalent correction as a sensitivity product.

The default analysis uses `MQF == 0` and GHSL `G7`, but neither is hard-coded. Change `ACCEPTED_MQF_VALUES` and `PRIMARY_GROUP` in the settings cell to rerun the complete workflow under another quality definition or GHSL group.

## RQ2 role

The notebook follows the required sequence: **observability first, geometry sensitivity second, recovery interpretation third**. Its inferential target is not whether VZA affects radiance in general, but whether residual VZA effects are large enough to alter the estimated Haiyan shock or recovery trajectory.

## Interpretation boundary

Baseline VZA–radiance relationships are diagnostic associations, not automatically valid correction functions. A correction is retained only as a sensitivity analysis unless it is supported across pixels, reduces orbit-structured variability, preserves observation support, and does not create implausible radiance values.

## 1. Quick literature tour: what has been tested for VZA

| Study | VZA contribution | Use in this notebook |
|---|---|---|
| [Román et al. (2018)](https://doi.org/10.1016/j.rse.2018.03.017) | Black Marble applies BRDF, lunar, atmospheric, terrain, snow, and quality corrections, but product correction does not guarantee application-level angular invariance. | Diagnose residual VZA effects after Black Marble processing. |
| [Li et al. (2019)](https://doi.org/10.1016/j.rse.2019.111357) | Demonstrated anisotropic artificial-light responses using repeated VIIRS observations and quadratic angular models. | Compare linear and quadratic within-pixel response forms. |
| [Wang et al. (2021)](https://doi.org/10.1016/j.rse.2021.112557) and [Wang et al. (2022)](https://doi.org/10.1109/LGRS.2022.3176616) | Quantified retrieval uncertainty and constructed all-angle, near-nadir, and off-nadir composites. | Report observation support, standard VZA intervals, and near- versus off-nadir contrasts. |
| [Mu et al. (2025)](https://doi.org/10.1109/TGRS.2024.3512549) | Found limited angular influence in an aggregated disaster-impact framework, partly because spatial aggregation and median filtering attenuated variability. | Test whether GHSL aggregation already suppresses VZA sensitivity before applying correction. |
| [Zheng et al. (2025)](https://doi.org/10.1016/j.rse.2025.114645) | Applied pre-event per-pixel quadratic VZA correction in post-hurricane recovery analysis; reported increasing bias at larger angles. | Produce a nadir-equivalent sensitivity series and compare disaster-phase estimates. |
| [Pei et al. (2025)](https://doi.org/10.5194/essd-17-5675-2025) | Used recurring VIIRS orbit groups to reduce angular effects and assessed remaining periodic information. | Measure the 16-day signal before and after correction. |
| [Tong (2025)](https://doi.org/10.48550/arXiv.2510.05105) | Showed that anisotropy varies by land use and brightness. | Stratify angular responses by GHSL settlement morphology; treat the preprint as supporting rather than definitive evidence. |

**Literature synthesis.** Existing work supports testing VZA, but not imposing a universal correction. The appropriate decision is empirical: retain all usable angles when the Haiyan result is stable; report thresholded sensitivity when phase estimates change; use correction only when it improves temporal behaviour without sacrificing observability.

### Analysis sequence

1. Define the MQF values, GHSL group, event window, VZA thresholds, and model support rules.
2. Load and align VNP46A1, VNP46A2, and GHSL.
3. Establish fixed baseline-lit spatial support and daily observability.
4. Describe VZA spatial fields, temporal evolution, settlement distributions, and the 16-day orbit cycle.
5. Estimate baseline daily and within-pixel VZA–radiance relationships.
6. Compare literature-standard VZA intervals and model-free VZA thresholds.
7. Fit quadratic response models and test a nadir-equivalent sensitivity product.
8. classify the result as geometry-robust, geometry-sensitive, or observation-limited.

In [10]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import rasterio

from rasterio.crs import CRS
from rasterio.transform import Affine
from rasterio.warp import reproject, Resampling

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from IPython.display import display

## 2. Settings and input paths

All analytical choices that are likely to change are defined here. The rest of the notebook derives labels, masks, filenames, and sensitivity tests from these settings.

To explore another reliability definition, change only:

- `ACCEPTED_MQF_VALUES`, such as `(0,)` or `(0, 1)`;
- `PRIMARY_GROUP`, from `G1` to `G8`; and
- the explicit VZA and support thresholds if required.

In [11]:
PROJECT_DIR = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_DIR = PROJECT_DIR / "datasets"
VNP46_DIR = DATA_DIR / "VNP46"
PROCESSED_DIR = VNP46_DIR / "processed"

A1_ZARR_PATH = PROCESSED_DIR / "Haiyan_VNP46A1.zarr"
A2_ZARR_PATH = PROCESSED_DIR / "Haiyan_VNP46A2.zarr"

GHSL_CANDIDATES = [
    VNP46_DIR / "GHSL_SMOD_E2015.tif",
    DATA_DIR / "ghsl" / "GHSL_SMOD_E2015.tif",
]

GHSL_PATH = next(
    (path for path in GHSL_CANDIDATES if path.exists()),
    GHSL_CANDIDATES[0],
)

EVENT_DATE = pd.Timestamp("2013-11-08")
ANALYSIS_START = EVENT_DATE - pd.Timedelta(days=180)
ANALYSIS_END = EVENT_DATE + pd.Timedelta(days=365)

PHASE_WINDOWS = [
    {"phase": "Baseline", "start_day": -180, "end_day": -1},
    {"phase": "Shock", "start_day": 0, "end_day": 30},
    {"phase": "Early recovery", "start_day": 31, "end_day": 90},
    {"phase": "Late recovery", "start_day": 91, "end_day": 180},
    {"phase": "Long-term", "start_day": 181, "end_day": 365},
]

PHASE_ORDER = [definition["phase"] for definition in PHASE_WINDOWS]

GHSL_CLASS_LABELS = {
    11: "Very low-density rural",
    12: "Low-density rural",
    13: "Rural cluster",
    21: "Suburban / peri-urban",
    22: "Semi-dense urban cluster",
    23: "Dense urban cluster",
    30: "Urban centre",
}

GHSL_CLASS_COLORS = {
    11: "#94A3B8",
    12: "#84CC16",
    13: "#22C55E",
    21: "#14B8A6",
    22: "#0EA5E9",
    23: "#6366F1",
    30: "#A855F7",
}

# Nested groups retained for comparability with RQ1.
GHSL_GROUPS = {
    "G1": (10, 11, 12, 13, 21, 22, 23, 30),
    "G2": (11, 12, 13, 21, 22, 23, 30),
    "G3": (12, 13, 21, 22, 23, 30),
    "G4": (13, 21, 22, 23, 30),
    "G5": (21, 22, 23, 30),
    "G6": (22, 23, 30),
    "G7": (23, 30),
    "G8": (30,),
}

# Primary analysis choices. These are defaults, not hard conditions.
PRIMARY_GROUP = "G7"
ACCEPTED_MQF_VALUES = (0,)

if PRIMARY_GROUP not in GHSL_GROUPS:
    raise ValueError(
        f"`PRIMARY_GROUP` must be one of {list(GHSL_GROUPS)}."
    )

if not ACCEPTED_MQF_VALUES:
    raise ValueError("Select at least one Mandatory Quality Flag value.")

PRIMARY_CODES = GHSL_GROUPS[PRIMARY_GROUP]
MQF_TEXT = ", ".join(str(value) for value in ACCEPTED_MQF_VALUES)
QUALITY_LABEL = (
    f"MQF = {MQF_TEXT}"
    if len(ACCEPTED_MQF_VALUES) == 1
    else f"MQF in {{{MQF_TEXT}}}"
)
OUTPUT_TAG = (
    f"{PRIMARY_GROUP.lower()}_mqf_"
    + "-".join(str(value) for value in ACCEPTED_MQF_VALUES)
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "outputs"
    / "haiyan_vza_effects"
    / OUTPUT_TAG
)
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# Fixed pre-Haiyan signal support.
BASELINE_MIN_OBSERVED_DAYS = 10
BASELINE_MIN_RADIANCE = 0.5

# Efficiency and reliability controls.
MAX_PIXELS_PER_CLASS = 150
RANDOM_SEED = 42
QC_SAMPLE_DAYS = 30
MIN_PIXEL_OBSERVATIONS = 10
MIN_PIXEL_VZA_SD = 2.0
MIN_DAILY_PIXELS = 10
VZA_MAX_PHYSICAL = 70.0
VZA_BINS = np.append(
    np.arange(0, VZA_MAX_PHYSICAL, 10),
    VZA_MAX_PHYSICAL + 0.001,
)
VZA_THRESHOLDS = [None, 50, 40, 30, 20]
STABILITY_THRESHOLDS = [40, 30]
ALL_VZA_SENTINEL = int(np.ceil(VZA_MAX_PHYSICAL)) + 100
VZA_ORBIT_EPOCH = pd.Timestamp("2012-01-19")
VZA_INTERVAL_EDGES = [0, 20, 40, 60, VZA_MAX_PHYSICAL + 0.001]
if VZA_MAX_PHYSICAL < 60:
    raise ValueError("`VZA_MAX_PHYSICAL` must be at least 60°.")

configured_thresholds = {
    threshold
    for threshold in VZA_THRESHOLDS
    if threshold is not None
}

if not set(STABILITY_THRESHOLDS).issubset(configured_thresholds):
    raise ValueError(
        "`STABILITY_THRESHOLDS` must be included in `VZA_THRESHOLDS`."
    )

if any(
    threshold > VZA_MAX_PHYSICAL
    for threshold in configured_thresholds
):
    raise ValueError(
        "VZA thresholds cannot exceed `VZA_MAX_PHYSICAL`."
    )

VZA_INTERVAL_LABELS = [
    "0-20° near-nadir",
    "20-40° middle",
    "40-60° off-nadir",
    f"60-{VZA_MAX_PHYSICAL:g}° extreme",
]

ROLLING_WINDOW = 14
BOOTSTRAP_REPETITIONS = 1_000

# Quadratic response support and correction diagnostics.
MIN_QUADRATIC_OBSERVATIONS = 20
MIN_QUADRATIC_VZA_SPAN = 40
MIN_REQUIRED_MAX_VZA = 50
MAX_MODEL_VZA = 60
IQR_MULTIPLIER = 1.5

# Phase-level sensitivity decision rules.
GEOMETRY_SENSITIVITY_PP = 5.0
MIN_PHASE_DATES_FOR_STABILITY = 10

# Exploratory screening rule: at least a 5% radiance change per 10°,
# with a bootstrap confidence interval for the median excluding zero.
MATERIAL_CHANGE_PCT_PER_10DEG = 5.0

HAIYAN_COLOR = "#2563EB"
DNB_COLOR = "#111827"
GAP_COLOR = "#F59E0B"
VZA_COLOR = "#7C3AED"

for required_path in [A1_ZARR_PATH, A2_ZARR_PATH, GHSL_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required input not found:\n{required_path.resolve()}"
        )

print("VNP46A1:", A1_ZARR_PATH)
print("VNP46A2:", A2_ZARR_PATH)
print("GHSL:", GHSL_PATH)
print("Analysis group:", PRIMARY_GROUP, PRIMARY_CODES)
print("Quality support:", QUALITY_LABEL)
print("Output variant:", OUTPUT_TAG)
print(
    "Analysis period:",
    ANALYSIS_START.strftime("%Y-%m-%d"),
    "to",
    ANALYSIS_END.strftime("%Y-%m-%d"),
)

VNP46A1: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/datasets/VNP46/processed/Haiyan_VNP46A1.zarr
VNP46A2: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/datasets/VNP46/processed/Haiyan_VNP46A2.zarr
GHSL: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/datasets/ghsl/GHSL_SMOD_E2015.tif
Analysis group: G7 (23, 30)
Quality support: MQF = 0
Output variant: g7_mqf_0
Analysis period: 2013-05-12 to 2014-11-08


## 3. Load and align VNP46A1 and VNP46A2

Notebook 00 may store observations on `processed`, `date`, or `time`. The helper normalises the acquisition date without loading the raster values, verifies the spatial grid, and reindexes both products to the complete event-centred calendar.

In [12]:
def parse_date_values(values):
    values = np.asarray(values)

    if np.issubdtype(values.dtype, np.datetime64):
        return pd.DatetimeIndex(pd.to_datetime(values)).normalize()

    date_strings = pd.Series(values.astype(str)).str.strip()

    if date_strings.str.fullmatch(r"\d{8}").all():
        parsed = pd.to_datetime(
            date_strings,
            format="%Y%m%d",
            errors="raise",
        )
    else:
        parsed = pd.to_datetime(date_strings, errors="raise")

    return pd.DatetimeIndex(parsed).normalize()


def normalise_date_axis(dataset, product_name):
    if "date" in dataset.dims:
        observation_dim = "date"
        raw_dates = dataset["date"].values

    elif "date" in dataset.variables:
        date_variable = dataset["date"]

        if len(date_variable.dims) != 1:
            raise ValueError(
                f"{product_name}: `date` must be one-dimensional; "
                f"found dimensions {date_variable.dims}."
            )

        observation_dim = date_variable.dims[0]
        raw_dates = date_variable.values

    elif "time" in dataset.dims:
        observation_dim = "time"
        raw_dates = dataset["time"].values

    else:
        raise KeyError(
            f"{product_name}: no usable acquisition-date variable. "
            f"Dimensions: {dict(dataset.sizes)}. "
            f"Variables: {list(dataset.variables)}"
        )

    parsed_dates = parse_date_values(raw_dates)

    if parsed_dates.isna().any():
        raise ValueError(
            f"{product_name}: at least one acquisition date could not be parsed."
        )

    dataset = dataset.assign_coords(
        date=(
            observation_dim,
            parsed_dates.to_numpy(dtype="datetime64[ns]"),
        )
    )

    if observation_dim != "date":
        dataset = dataset.swap_dims({observation_dim: "date"})

    dataset = dataset.sortby("date")
    date_index = pd.DatetimeIndex(dataset["date"].values)

    duplicate_dates = date_index[date_index.duplicated(keep=False)]

    if len(duplicate_dates) > 0:
        duplicate_text = ", ".join(
            pd.Index(duplicate_dates.strftime("%Y-%m-%d"))
            .unique()
            .tolist()[:10]
        )
        raise ValueError(
            f"{product_name}: duplicate dates remain after Notebook 00 "
            f"processing: {duplicate_text}"
        )

    return dataset


vnp46a1 = normalise_date_axis(
    xr.open_zarr(A1_ZARR_PATH, chunks="auto"),
    "VNP46A1",
)

vnp46a2 = normalise_date_axis(
    xr.open_zarr(A2_ZARR_PATH, chunks="auto"),
    "VNP46A2",
)

print("VNP46A1 dimensions:", dict(vnp46a1.sizes))
print("VNP46A2 dimensions:", dict(vnp46a2.sizes))

VNP46A1 dimensions: {'date': 546, 'y': 674, 'x': 473}
VNP46A2 dimensions: {'date': 546, 'y': 674, 'x': 473}


In [13]:
REQUIRED_A1_VARIABLES = [
    "Sensor_Zenith",
]

REQUIRED_A2_VARIABLES = [
    "DNB_BRDF_Corrected_NTL",
    "Gap_Filled_DNB_BRDF_Corrected_NTL",
    "Mandatory_Quality_Flag",
]

missing_a1 = [
    variable
    for variable in REQUIRED_A1_VARIABLES
    if variable not in vnp46a1
]

missing_a2 = [
    variable
    for variable in REQUIRED_A2_VARIABLES
    if variable not in vnp46a2
]

if missing_a1:
    raise KeyError(
        f"Missing VNP46A1 variables: {missing_a1}\n"
        f"Available variables: {list(vnp46a1.data_vars)}"
    )

if missing_a2:
    raise KeyError(
        f"Missing VNP46A2 variables: {missing_a2}\n"
        f"Available variables: {list(vnp46a2.data_vars)}"
    )

for coordinate in ["x", "y"]:
    if coordinate not in vnp46a1.coords:
        raise KeyError(f"VNP46A1 is missing `{coordinate}`.")

    if coordinate not in vnp46a2.coords:
        raise KeyError(f"VNP46A2 is missing `{coordinate}`.")

if not np.allclose(vnp46a1["x"].values, vnp46a2["x"].values):
    raise ValueError("VNP46A1 and VNP46A2 x coordinates do not match.")

if not np.allclose(vnp46a1["y"].values, vnp46a2["y"].values):
    raise ValueError("VNP46A1 and VNP46A2 y coordinates do not match.")

full_dates = pd.date_range(
    ANALYSIS_START,
    ANALYSIS_END,
    freq="D",
)

vnp46a1 = (
    vnp46a1
    .sel(date=slice(ANALYSIS_START, ANALYSIS_END))
    .reindex(date=full_dates)
)

vnp46a2 = (
    vnp46a2
    .sel(date=slice(ANALYSIS_START, ANALYSIS_END))
    .reindex(date=full_dates)
)

print("Expected calendar days:", len(full_dates))
print("VNP46A1 dates after reindexing:", vnp46a1.sizes["date"])
print("VNP46A2 dates after reindexing:", vnp46a2.sizes["date"])

Expected calendar days: 546
VNP46A1 dates after reindexing: 546
VNP46A2 dates after reindexing: 546


## 4.1 Clean the VZA, radiance, and quality bands

VZA is converted from the native 0–9000 integer representation only when necessary. Direct DNB-BRDF is retained on the selected `ACCEPTED_MQF_VALUES`; paired gap-filled values use exactly the same pixel-days. The operational gap-filled band is preserved separately.

In [14]:
def clean_band(data_array):
    cleaned = data_array.squeeze(drop=True).astype("float32")

    for attribute in ["_FillValue", "missing_value", "nodata"]:
        fill_value = data_array.attrs.get(attribute)

        if fill_value is None:
            continue

        for value in np.atleast_1d(fill_value):
            try:
                cleaned = cleaned.where(cleaned != float(value))
            except (TypeError, ValueError):
                pass

    return cleaned.where(np.isfinite(cleaned))


dnb = clean_band(
    vnp46a2["DNB_BRDF_Corrected_NTL"]
).where(
    lambda values: (values >= 0) & (values <= 6553.4)
)

gap_filled = clean_band(
    vnp46a2["Gap_Filled_DNB_BRDF_Corrected_NTL"]
).where(
    lambda values: (values >= 0) & (values <= 6553.4)
)

mqf = clean_band(
    vnp46a2["Mandatory_Quality_Flag"]
)

sensor_zenith = clean_band(
    vnp46a1["Sensor_Zenith"]
)

qc_date_slice = slice(
    0,
    min(QC_SAMPLE_DAYS, sensor_zenith.sizes["date"]),
)

vza_sample_max = (
    sensor_zenith
    .isel(date=qc_date_slice)
    .max(skipna=True)
    .compute()
    .item()
)

if np.isfinite(vza_sample_max) and vza_sample_max > 180:
    sensor_zenith = sensor_zenith * 0.01

sensor_zenith = sensor_zenith.where(
    (sensor_zenith >= 0) & (sensor_zenith <= 90)
)

quality_mask = mqf.isin(list(ACCEPTED_MQF_VALUES))
qualified_dnb = dnb.where(quality_mask)

qc_reductions = xr.Dataset(
    {
        "vza_min": sensor_zenith.isel(date=qc_date_slice).min(skipna=True),
        "vza_max": sensor_zenith.isel(date=qc_date_slice).max(skipna=True),
        "dnb_min": dnb.isel(date=qc_date_slice).min(skipna=True),
        "dnb_max": dnb.isel(date=qc_date_slice).max(skipna=True),
        "gap_min": gap_filled.isel(date=qc_date_slice).min(skipna=True),
        "gap_max": gap_filled.isel(date=qc_date_slice).max(skipna=True),
        "mqf_min": mqf.isel(date=qc_date_slice).min(skipna=True),
        "mqf_max": mqf.isel(date=qc_date_slice).max(skipna=True),
    }
).compute()

band_qc = pd.DataFrame(
    {
        "band": [
            "Sensor_Zenith",
            "DNB_BRDF_Corrected_NTL",
            "Gap_Filled_DNB_BRDF_Corrected_NTL",
            "Mandatory_Quality_Flag",
        ],
        "scope": [f"First {QC_SAMPLE_DAYS} days"] * 4,
        "minimum": [
            qc_reductions["vza_min"].item(),
            qc_reductions["dnb_min"].item(),
            qc_reductions["gap_min"].item(),
            qc_reductions["mqf_min"].item(),
        ],
        "maximum": [
            qc_reductions["vza_max"].item(),
            qc_reductions["dnb_max"].item(),
            qc_reductions["gap_max"].item(),
            qc_reductions["mqf_max"].item(),
        ],
    }
)

display(band_qc.style.format({"minimum": "{:.2f}", "maximum": "{:.2f}"}))

,band,scope,minimum,maximum
0,Sensor_Zenith,First 30 days,0.02,66.33
1,DNB_BRDF_Corrected_NTL,First 30 days,0.00,105.06
2,Gap_Filled_DNB_BRDF_Corrected_NTL,First 30 days,0.00,105.06
3,Mandatory_Quality_Flag,First 30 days,0.00,5.00


**Table 1 — Input-band quality check.** The table confirms the decoded ranges of VZA, direct DNB-BRDF, gap-filled radiance, and MQF over the initial sample days.

**RQ2 relevance:** implausible ranges indicate scaling, fill-value, or band-alignment errors that must be resolved before angular or recovery interpretation.

## 4.2 Align GHSL to the VNP46 grid

Atomic GHSL classes support direct settlement comparisons. Nested G1–G8 groups are retained for RQ1 continuity; `PRIMARY_GROUP` selects the group used for representative maps, threshold stability, and correction sensitivity.

In [15]:
def get_dataset_crs(dataset):
    if "spatial_ref" in dataset.variables:
        attributes = dataset["spatial_ref"].attrs

        for key in ["crs_wkt", "spatial_ref"]:
            value = attributes.get(key)

            if value:
                return CRS.from_user_input(value)

        epsg_code = attributes.get("epsg_code")

        if epsg_code:
            return CRS.from_user_input(epsg_code)

    for key in ["crs", "crs_wkt"]:
        value = dataset.attrs.get(key)

        if value:
            return CRS.from_user_input(value)

    x_values = dataset["x"].values
    y_values = dataset["y"].values

    if (
        np.nanmin(x_values) >= -180
        and np.nanmax(x_values) <= 180
        and np.nanmin(y_values) >= -90
        and np.nanmax(y_values) <= 90
    ):
        return CRS.from_epsg(4326)

    raise ValueError(
        "The VNP46 CRS is missing from the processed Zarr metadata."
    )


def get_dataset_transform(dataset):
    x_values = np.asarray(dataset["x"].values, dtype=float)
    y_values = np.asarray(dataset["y"].values, dtype=float)

    if len(x_values) < 2 or len(y_values) < 2:
        raise ValueError("At least two x and y coordinates are required.")

    x_resolution = float(np.median(np.diff(x_values)))
    y_resolution = float(np.median(np.diff(y_values)))

    return Affine(
        x_resolution,
        0,
        x_values[0] - x_resolution / 2,
        0,
        y_resolution,
        y_values[0] - y_resolution / 2,
    )


reference_crs = get_dataset_crs(vnp46a2)
reference_transform = get_dataset_transform(vnp46a2)

ghsl_aligned = np.full(
    (vnp46a2.sizes["y"], vnp46a2.sizes["x"]),
    np.nan,
    dtype="float32",
)

with rasterio.open(GHSL_PATH) as source:
    reproject(
        source=rasterio.band(source, 1),
        destination=ghsl_aligned,
        src_transform=source.transform,
        src_crs=source.crs,
        src_nodata=source.nodata,
        dst_transform=reference_transform,
        dst_crs=reference_crs,
        dst_nodata=np.nan,
        resampling=Resampling.nearest,
    )

ghsl = xr.DataArray(
    ghsl_aligned,
    dims=("y", "x"),
    coords={"y": vnp46a2["y"], "x": vnp46a2["x"]},
    name="GHSL_SMOD_E2015",
)

available_classes = np.unique(
    ghsl_aligned[np.isfinite(ghsl_aligned)]
).astype(int)

missing_classes = sorted(
    set(GHSL_CLASS_LABELS) - set(available_classes)
)

print("VNP46 CRS:", reference_crs)
print("Aligned GHSL classes:", available_classes)

if missing_classes:
    print("Settlement classes absent from the aligned AOI:", missing_classes)

VNP46 CRS: EPSG:4326
Aligned GHSL classes: [ 0 10 11 12 13 21 22 23 30]


**GHSL alignment check.** The printed CRS and class inventory verify that nearest-neighbour reprojection preserved categorical settlement codes.

**RQ2 relevance:** a missing or shifted settlement class would confound morphology with geometry and invalidate group-specific recovery comparisons.

## 4.3 Define fixed pre-Haiyan settlement-light support

The fixed mask prevents a changing set of lit pixels from being mistaken for a VZA response. A pixel must meet `BASELINE_MIN_OBSERVED_DAYS` under the selected MQF definition and exceed `BASELINE_MIN_RADIANCE`. Both thresholds remain editable in the settings cell.

In [16]:
baseline_slice = slice(
    ANALYSIS_START,
    EVENT_DATE - pd.Timedelta(days=1),
)

baseline_qualified = qualified_dnb.sel(date=baseline_slice)

baseline_gap_paired = gap_filled.where(
    qualified_dnb.notnull()
).sel(date=baseline_slice)

baseline_reductions = xr.Dataset(
    {
        "observed_days": baseline_qualified.notnull().sum(dim="date"),
        "median_dnb": baseline_qualified.median(
            dim="date",
            skipna=True,
        ),
        "median_gap_paired": baseline_gap_paired.median(
            dim="date",
            skipna=True,
        ),
        "median_gap_all": (
            gap_filled
            .sel(date=baseline_slice)
            .median(dim="date", skipna=True)
        ),
    }
).compute()

baseline_observed_days = baseline_reductions["observed_days"]
baseline_median_dnb = baseline_reductions["median_dnb"]
baseline_median_gap_paired = baseline_reductions["median_gap_paired"]
baseline_median_gap_all = baseline_reductions["median_gap_all"]

signal_mask = (
    (baseline_observed_days >= BASELINE_MIN_OBSERVED_DAYS)
    & (baseline_median_dnb > BASELINE_MIN_RADIANCE)
)

support_records = []

for class_code, class_label in GHSL_CLASS_LABELS.items():
    class_mask = ghsl == class_code
    support_mask = class_mask & signal_mask

    support_records.append(
        {
            "class_code": class_code,
            "settlement_class": class_label,
            "ghsl_pixels": int(class_mask.sum().item()),
            "signal_pixels": int(support_mask.sum().item()),
        }
    )

support_summary = pd.DataFrame(support_records)
support_summary["retained_pct"] = (
    100
    * support_summary["signal_pixels"]
    / support_summary["ghsl_pixels"].replace(0, np.nan)
)

display(
    support_summary.style.format(
        {
            "ghsl_pixels": "{:,.0f}",
            "signal_pixels": "{:,.0f}",
            "retained_pct": "{:.1f}",
        },
        na_rep="—",
    )
)

,class_code,settlement_class,ghsl_pixels,signal_pixels,retained_pct
0,11,Very low-density rural,"64,310",114,0.2
1,12,Low-density rural,"16,706",227,1.4
2,13,Rural cluster,"4,356",123,2.8
3,21,Suburban / peri-urban,"11,988",945,7.9
4,22,Semi-dense urban cluster,"1,770",266,15.0
5,23,Dense urban cluster,"2,060",945,45.9
6,30,Urban centre,622,497,79.9


**Table 2 — Fixed-support inventory.** `retained_pct` reports how much of each GHSL class remains after baseline observation-count and radiance screening.

**RQ2 relevance:** classes with very small or weakly retained support may be computable but not interpretable. This table separates a missing recovery signal from a settlement class that was never adequately observed.

## 5.1 Full-grid daily observability and radiance summaries

All reductions are assembled into one lazy xarray computation. Direct and paired gap-filled radiance use the same selected-MQF pixel-days. The operational gap-filled series uses every available gap-filled value within the fixed mask and remains analytically separate.

In [17]:
spatial_dimensions = ["y", "x"]
daily_variables = {}
class_pixel_counts = {}

for class_code in GHSL_CLASS_LABELS:
    class_mask = (ghsl == class_code) & signal_mask
    class_pixel_count = int(class_mask.sum().item())
    class_pixel_counts[class_code] = class_pixel_count

    if class_pixel_count == 0:
        continue

    prefix = f"c{class_code}"
    direct_class = qualified_dnb.where(class_mask)
    gap_paired_class = gap_filled.where(direct_class.notnull())
    gap_all_class = gap_filled.where(class_mask)
    vza_paired_class = sensor_zenith.where(direct_class.notnull())

    direct_relative = (
        direct_class
        / baseline_median_dnb.where(class_mask)
        - 1
    )

    gap_paired_relative = (
        gap_paired_class
        / baseline_median_gap_paired.where(class_mask)
        - 1
    )

    gap_all_relative = (
        gap_all_class
        / baseline_median_gap_all.where(class_mask)
        - 1
    )

    daily_variables[f"{prefix}__observed_pixels"] = (
        direct_class.notnull().sum(dim=spatial_dimensions)
    )

    daily_variables[f"{prefix}__vza_mean"] = (
        vza_paired_class.mean(dim=spatial_dimensions, skipna=True)
    )

    daily_variables[f"{prefix}__vza_median"] = (
        vza_paired_class.median(dim=spatial_dimensions, skipna=True)
    )

    daily_variables[f"{prefix}__vza_min"] = (
        vza_paired_class.min(dim=spatial_dimensions, skipna=True)
    )

    daily_variables[f"{prefix}__vza_max"] = (
        vza_paired_class.max(dim=spatial_dimensions, skipna=True)
    )

    daily_variables[f"{prefix}__dnb_mean"] = (
        direct_class.mean(dim=spatial_dimensions, skipna=True)
    )

    daily_variables[f"{prefix}__gap_paired_mean"] = (
        gap_paired_class.mean(dim=spatial_dimensions, skipna=True)
    )

    daily_variables[f"{prefix}__gap_all_mean"] = (
        gap_all_class.mean(dim=spatial_dimensions, skipna=True)
    )

    daily_variables[f"{prefix}__dnb_relative_median"] = (
        direct_relative.median(dim=spatial_dimensions, skipna=True)
    )

    daily_variables[f"{prefix}__gap_paired_relative_median"] = (
        gap_paired_relative.median(dim=spatial_dimensions, skipna=True)
    )

    daily_variables[f"{prefix}__gap_all_relative_median"] = (
        gap_all_relative.median(dim=spatial_dimensions, skipna=True)
    )

daily_reductions = xr.Dataset(daily_variables).compute()

In [18]:
def assign_phase(frame):
    frame = frame.copy()
    frame["relative_day"] = (frame["date"] - EVENT_DATE).dt.days
    frame["phase"] = pd.NA

    for definition in PHASE_WINDOWS:
        phase_mask = frame["relative_day"].between(
            definition["start_day"],
            definition["end_day"],
            inclusive="both",
        )
        frame.loc[phase_mask, "phase"] = definition["phase"]

    frame["phase"] = pd.Categorical(
        frame["phase"],
        categories=PHASE_ORDER,
        ordered=True,
    )

    return frame


daily_frames = []
daily_dates = pd.DatetimeIndex(daily_reductions["date"].values)

for class_code, class_label in GHSL_CLASS_LABELS.items():
    prefix = f"c{class_code}"

    if f"{prefix}__vza_median" not in daily_reductions:
        continue

    class_frame = pd.DataFrame(
        {
            "date": daily_dates,
            "class_code": class_code,
            "settlement_class": class_label,
            "signal_pixels": class_pixel_counts[class_code],
            "observed_pixels": daily_reductions[
                f"{prefix}__observed_pixels"
            ].values,
            "vza_mean": daily_reductions[
                f"{prefix}__vza_mean"
            ].values,
            "vza_median": daily_reductions[
                f"{prefix}__vza_median"
            ].values,
            "vza_min": daily_reductions[
                f"{prefix}__vza_min"
            ].values,
            "vza_max": daily_reductions[
                f"{prefix}__vza_max"
            ].values,
            "dnb_mean": daily_reductions[
                f"{prefix}__dnb_mean"
            ].values,
            "gap_paired_mean": daily_reductions[
                f"{prefix}__gap_paired_mean"
            ].values,
            "gap_all_mean": daily_reductions[
                f"{prefix}__gap_all_mean"
            ].values,
            "dnb_relative_median": daily_reductions[
                f"{prefix}__dnb_relative_median"
            ].values,
            "gap_paired_relative_median": daily_reductions[
                f"{prefix}__gap_paired_relative_median"
            ].values,
            "gap_all_relative_median": daily_reductions[
                f"{prefix}__gap_all_relative_median"
            ].values,
        }
    )

    class_frame["valid_pct"] = (
        100
        * class_frame["observed_pixels"]
        / class_frame["signal_pixels"]
    )

    daily_frames.append(class_frame)

daily_summary = assign_phase(
    pd.concat(daily_frames, ignore_index=True)
)

daily_summary = (
    daily_summary
    .sort_values(["class_code", "date"])
    .reset_index(drop=True)
)

daily_summary.to_csv(
    TABLE_DIR / "daily_vza_settlement_summary.csv",
    index=False,
)

display(
    daily_summary.head().style.format(
        {
            "vza_mean": "{:.2f}",
            "vza_median": "{:.2f}",
            "vza_min": "{:.2f}",
            "vza_max": "{:.2f}",
            "dnb_mean": "{:.3f}",
            "gap_paired_mean": "{:.3f}",
            "gap_all_mean": "{:.3f}",
            "dnb_relative_median": "{:.3f}",
            "gap_paired_relative_median": "{:.3f}",
            "gap_all_relative_median": "{:.3f}",
            "valid_pct": "{:.1f}",
        },
        na_rep="—",
    )
)

,date,class_code,settlement_class,signal_pixels,observed_pixels,vza_mean,vza_median,vza_min,vza_max,dnb_mean,gap_paired_mean,gap_all_mean,dnb_relative_median,gap_paired_relative_median,gap_all_relative_median,valid_pct,relative_day,phase
0,2013-05-12 00:00:00,11,Very low-density rural,114,1,8.18,8.18,8.18,8.18,0.642,0.642,1.969,0.183,0.183,0.048,0.9,-180,Baseline
1,2013-05-13 00:00:00,11,Very low-density rural,114,0,—,—,—,—,—,—,1.968,—,—,0.045,0.0,-179,Baseline
2,2013-05-14 00:00:00,11,Very low-density rural,114,5,52.87,52.81,51.96,53.83,1.327,1.327,2.007,0.956,0.956,0.057,4.4,-178,Baseline
3,2013-05-15 00:00:00,11,Very low-density rural,114,112,63.75,63.57,62.09,65.76,3.250,3.250,3.285,0.665,0.665,0.911,98.2,-177,Baseline
4,2013-05-16 00:00:00,11,Very low-density rural,114,107,46.97,46.76,44.35,50.53,2.671,2.671,2.580,0.388,0.388,0.445,93.9,-176,Baseline


**Table 3 — Daily settlement summary preview.** The full CSV records VZA, observed pixels, valid-pixel percentage, direct DNB-BRDF, paired gap-filled, operational gap-filled, phase, and event-relative day.

**RQ2 relevance:** every later angular or recovery comparison can be traced to phase, settlement class, product support, and daily observability.

## 5.2 Representative VZA fields

Three dates are selected from the 10th, 50th, and 90th percentiles of the selected GHSL group's daily median VZA distribution, subject to `MIN_DAILY_PIXELS`. The maps show the complete A1 VZA field; later outputs quantify settlement-specific support.

In [19]:
primary_codes = PRIMARY_CODES
primary_daily = (
    daily_summary.loc[
        daily_summary["class_code"].isin(primary_codes)
    ]
    .groupby("date", as_index=False)
    .agg(
        vza_median=("vza_median", "median"),
        observed_pixels=("observed_pixels", "sum"),
        signal_pixels=("signal_pixels", "sum"),
    )
)

primary_daily["valid_pct"] = (
    100
    * primary_daily["observed_pixels"]
    / primary_daily["signal_pixels"]
)

representative_candidates = (
    primary_daily
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["vza_median"])
    .loc[lambda frame: frame["observed_pixels"] >= MIN_DAILY_PIXELS]
)

if representative_candidates.empty:
    raise ValueError(
        f"No {PRIMARY_GROUP} dates satisfy the representative-map requirements."
    )

representative_records = []

for quantile_label, quantile_value in [
    ("Low VZA", 0.10),
    ("Median VZA", 0.50),
    ("High VZA", 0.90),
]:
    target = representative_candidates["vza_median"].quantile(
        quantile_value
    )

    selected_index = (
        representative_candidates["vza_median"] - target
    ).abs().idxmin()

    selected = representative_candidates.loc[selected_index]

    representative_records.append(
        {
            "label": quantile_label,
            "date": pd.Timestamp(selected["date"]),
            "vza_median": selected["vza_median"],
            "valid_pct": selected["valid_pct"],
        }
    )

representative_dates = pd.DataFrame(representative_records)
display(
    representative_dates.style.format(
        {"vza_median": "{:.2f}", "valid_pct": "{:.1f}"}
    )
)

,label,date,vza_median,valid_pct
0,Low VZA,2014-03-23 00:00:00,9.07,1.9
1,Median VZA,2014-04-09 00:00:00,42.40,12.9
2,High VZA,2013-09-04 00:00:00,63.40,43.2


In [20]:
raster_x = vnp46a1["x"].values
raster_y = vnp46a1["y"].values

selected_map_dates = (
    pd.DatetimeIndex(representative_dates["date"])
    .unique()
    .to_list()
)

selected_vza_maps = (
    sensor_zenith
    .sel(date=selected_map_dates)
    .compute()
)

fig = make_subplots(
    rows=1,
    cols=3,
    horizontal_spacing=0.055,
    subplot_titles=[
        (
            f"<b>{record.label}</b><br>"
            f"{record.date:%Y-%m-%d}; "
            f"{PRIMARY_GROUP} median = {record.vza_median:.1f}°"
        )
        for record in representative_dates.itertuples()
    ],
)

for column, record in enumerate(
    representative_dates.itertuples(),
    start=1,
):
    fig.add_trace(
        go.Heatmap(
            x=raster_x,
            y=raster_y,
            z=selected_vza_maps.sel(date=record.date).values,
            zmin=0,
            zmax=70,
            colorscale="Viridis",
            showscale=column == 3,
            colorbar={
                "title": {"text": "VZA (°)"},
                "len": 0.70,
                "thickness": 16,
            },
            hovertemplate=(
                "Longitude: %{x:.4f}°<br>"
                "Latitude: %{y:.4f}°<br>"
                "VZA: %{z:.2f}°"
                "<extra></extra>"
            ),
        ),
        row=1,
        col=column,
    )

    fig.update_xaxes(
        title_text="Longitude (°E)",
        constrain="domain",
        row=1,
        col=column,
    )

    fig.update_yaxes(
        title_text="Latitude (°N)" if column == 1 else None,
        scaleanchor=f"x{column}" if column > 1 else "x",
        scaleratio=1,
        row=1,
        col=column,
    )

fig.update_layout(
    title={
        "text": "Representative VIIRS viewing zenith angle fields",
        "x": 0.01,
        "xanchor": "left",
    },
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    font={"family": "Arial", "size": 16, "color": "#111827"},
    margin={"l": 75, "r": 100, "t": 120, "b": 70},
    width=1550,
    height=610,
)

fig.show()
fig.write_html(
    FIGURE_DIR / "01_representative_vza_fields.html",
    include_plotlyjs="cdn",
)

**Table 4 — Representative dates.** The table records the selected date, group-level median VZA, and valid-pixel percentage. It verifies that low-, middle-, and high-angle maps remain sufficiently observed.

**Figure 1 — Representative VZA fields.** A coherent cross-track gradient indicates orbital geometry; irregular holes indicate retrieval loss or changing spatial support.

**RQ2 relevance:** spatially patchy support should be treated as an observability limitation, not as evidence that VZA caused a radiance change.

## 5.3 VZA evolution and observability by settlement

Daily median VZA is shown with a centred rolling median. The lower panel retains valid-pixel coverage so that changes in viewing geometry can be separated from changes in observed spatial support.

In [21]:
daily_summary["vza_median_14d"] = (
    daily_summary
    .groupby("class_code", observed=True)["vza_median"]
    .transform(
        lambda values: values.rolling(
            window=ROLLING_WINDOW,
            center=True,
            min_periods=3,
        ).median()
    )
)

daily_summary["valid_pct_14d"] = (
    daily_summary
    .groupby("class_code", observed=True)["valid_pct"]
    .transform(
        lambda values: values.rolling(
            window=ROLLING_WINDOW,
            center=True,
            min_periods=3,
        ).median()
    )
)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.10,
    row_heights=[0.62, 0.38],
    subplot_titles=[
        "Daily median VZA by GHSL settlement class",
        f"Selected-quality DNB-BRDF spatial coverage ({QUALITY_LABEL})",
    ],
)

for class_code, class_label in GHSL_CLASS_LABELS.items():
    class_data = daily_summary.loc[
        daily_summary["class_code"] == class_code
    ]

    if class_data.empty:
        continue

    color = GHSL_CLASS_COLORS[class_code]

    fig.add_trace(
        go.Scatter(
            x=class_data["date"],
            y=class_data["vza_median_14d"],
            mode="lines",
            name=class_label,
            legendgroup=str(class_code),
            line={"color": color, "width": 2.2},
            customdata=np.column_stack(
                [
                    class_data["vza_median"],
                    class_data["valid_pct"],
                ]
            ),
            hovertemplate=(
                "%{x|%Y-%m-%d}<br>"
                "14-day median VZA: %{y:.2f}°<br>"
                "Daily median VZA: %{customdata[0]:.2f}°<br>"
                "Valid pixels: %{customdata[1]:.1f}%"
                "<extra></extra>"
            ),
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=class_data["date"],
            y=class_data["valid_pct_14d"],
            mode="lines",
            name=class_label,
            legendgroup=str(class_code),
            showlegend=False,
            line={"color": color, "width": 2.0},
            hovertemplate=(
                "%{x|%Y-%m-%d}<br>"
                "14-day median coverage: %{y:.1f}%"
                "<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

for row in [1, 2]:
    fig.add_vline(
        x=EVENT_DATE.to_pydatetime(),
        line={"color": HAIYAN_COLOR, "width": 2, "dash": "dash"},
        row=row,
        col=1,
    )

fig.add_annotation(
    x=EVENT_DATE.to_pydatetime(),
    y=1.04,
    xref="x",
    yref="paper",
    text="Haiyan/Yolanda<br>8 Nov 2013",
    showarrow=False,
    xanchor="left",
    yanchor="bottom",
    font={"size": 13, "color": HAIYAN_COLOR},
    bgcolor="rgba(255,255,255,0.88)",
)

fig.update_yaxes(title_text="Sensor zenith angle (°)", row=1, col=1)
fig.update_yaxes(title_text=f"{QUALITY_LABEL} pixels (%)", range=[0, 100], row=2, col=1)
fig.update_xaxes(title_text="Date", row=2, col=1)

fig.update_layout(
    title={
        "text": "Viewing geometry and observability across settlement classes",
        "x": 0.01,
        "xanchor": "left",
    },
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    font={"family": "Arial", "size": 16, "color": "#111827"},
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.12,
        "xanchor": "right",
        "x": 1,
    },
    margin={"l": 95, "r": 50, "t": 165, "b": 75},
    width=1500,
    height=850,
)

fig.show()
fig.write_html(
    FIGURE_DIR / "02_vza_time_by_settlement.html",
    include_plotlyjs="cdn",
)

**Figure 2 — VZA evolution and observability.** The upper panel shows whether phase-specific radiance changes coincide with systematic VZA shifts; the lower panel shows whether they also coincide with loss of usable pixels.

**RQ2 relevance:** a radiance decline is more interpretable as disaster impact when it is not explained by extreme geometry or severe observation loss. Otherwise, the interval should be classified as observation-limited.

## 5.4 Settlement-level VZA distributions

Each observation is a daily settlement-class median, preventing larger classes from dominating through pixel count alone. Baseline and long-term windows are compared because shock and recovery contain real radiance change and are not suitable for estimating an unconfounded VZA effect.

In [22]:
distribution_data = (
    daily_summary.loc[
        daily_summary["phase"].isin(["Baseline", "Long-term"])
        & (daily_summary["observed_pixels"] >= MIN_DAILY_PIXELS)
    ]
    .dropna(subset=["vza_median"])
    .copy()
)

vza_distribution_summary = (
    distribution_data
    .groupby(
        ["class_code", "settlement_class", "phase"],
        observed=True,
    )
    .agg(
        dates=("vza_median", "size"),
        vza_p10=("vza_median", lambda values: values.quantile(0.10)),
        vza_median=("vza_median", "median"),
        vza_p90=("vza_median", lambda values: values.quantile(0.90)),
        median_valid_pct=("valid_pct", "median"),
    )
    .reset_index()
)

display(
    vza_distribution_summary.style.format(
        {
            "vza_p10": "{:.2f}",
            "vza_median": "{:.2f}",
            "vza_p90": "{:.2f}",
            "median_valid_pct": "{:.1f}",
        }
    )
)

,class_code,settlement_class,phase,dates,vza_p10,vza_median,vza_p90,median_valid_pct
0,11,Very low-density rural,Baseline,64,9.31,44.11,63.44,67.5
1,11,Very low-density rural,Long-term,87,8.76,39.69,63.29,64.0
2,12,Low-density rural,Baseline,79,17.90,41.72,63.58,43.6
3,12,Low-density rural,Long-term,108,7.40,40.08,63.42,46.0
4,13,Rural cluster,Baseline,73,18.96,41.80,63.70,52.0
5,13,Rural cluster,Long-term,97,8.34,39.61,63.82,50.4
6,21,Suburban / peri-urban,Baseline,90,17.75,42.87,63.62,36.1
7,21,Suburban / peri-urban,Long-term,118,7.13,40.31,63.26,37.9
8,22,Semi-dense urban cluster,Baseline,79,18.03,41.65,63.72,45.1
9,22,Semi-dense urban cluster,Long-term,111,7.99,40.61,63.59,39.8


**Table 5 — VZA distribution by settlement and stable window.** The table reports the number of dates, P10, median, P90, and median valid-pixel percentage.

**RQ2 relevance:** phase or settlement comparisons are strongest when they share comparable VZA ranges and adequate observation support.

In [23]:
fig = go.Figure()

for phase_name, line_color in [
    ("Baseline", "#334155"),
    ("Long-term", "#16A34A"),
]:
    phase_data = distribution_data.loc[
        distribution_data["phase"] == phase_name
    ]

    fig.add_trace(
        go.Box(
            x=phase_data["settlement_class"],
            y=phase_data["vza_median"],
            name=phase_name,
            marker_color=line_color,
            boxmean=True,
            boxpoints="outliers",
            jitter=0.25,
            pointpos=0,
            hovertemplate=(
                "Settlement: %{x}<br>"
                "Daily median VZA: %{y:.2f}°"
                "<extra></extra>"
            ),
        )
    )

fig.update_yaxes(title_text="Daily median sensor zenith angle (°)")
fig.update_xaxes(title_text="GHSL settlement class", tickangle=-20)

fig.update_layout(
    title={
        "text": "VZA distributions by settlement class and stable analysis window",
        "x": 0.01,
        "xanchor": "left",
    },
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    font={"family": "Arial", "size": 16, "color": "#111827"},
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.04,
        "xanchor": "right",
        "x": 1,
    },
    margin={"l": 90, "r": 45, "t": 115, "b": 145},
    width=1450,
    height=680,
)

fig.show()
fig.write_html(
    FIGURE_DIR / "03_vza_distribution_by_settlement.html",
    include_plotlyjs="cdn",
)

**Figure 3 — VZA distributions by settlement class.** Overlapping baseline and long-term boxes indicate comparable geometry; systematic displacement indicates that temporal comparisons may inherit a geometry difference.

**RQ2 relevance:** settlement-specific recovery trajectories should be qualified when their observation-angle distributions differ materially.

## 5.5 Sixteen-day VIIRS orbit-cycle diagnostic

VIIRS viewing geometry repeats systematically with the orbit. The baseline window is used to estimate lag-16 correlation and the fraction of VZA variation explained by day within the cycle.

In [24]:
orbit_daily = daily_summary.loc[
    (daily_summary["phase"] == "Baseline")
    & daily_summary["vza_mean"].between(0, VZA_MAX_PHYSICAL)
].copy()

orbit_daily["orbit_day"] = (
    (orbit_daily["date"] - VZA_ORBIT_EPOCH).dt.days % 16
) + 1

orbit_records = []

for (class_code, settlement_class), group in orbit_daily.groupby(
    ["class_code", "settlement_class"],
    observed=True,
):
    group = group.sort_values("date").copy()

    complete_vza = (
        group.set_index("date")["vza_mean"]
        .reindex(
            pd.date_range(
                group["date"].min(),
                group["date"].max(),
                freq="D",
            )
        )
    )

    lag16_r = complete_vza.corr(complete_vza.shift(16))
    orbit_means = group.groupby("orbit_day")["vza_mean"].mean()
    fitted = group["orbit_day"].map(orbit_means)

    total_ss = np.square(
        group["vza_mean"] - group["vza_mean"].mean()
    ).sum()
    residual_ss = np.square(group["vza_mean"] - fitted).sum()

    orbit_records.append(
        {
            "class_code": class_code,
            "settlement_class": settlement_class,
            "baseline_days": group["date"].nunique(),
            "lag16_vza_r": lag16_r,
            "orbit_day_vza_r2": (
                1 - residual_ss / total_ss
                if total_ss > 0
                else np.nan
            ),
            "vza_range_degrees": (
                group["vza_mean"].max() - group["vza_mean"].min()
            ),
        }
    )

orbit_diagnostics = pd.DataFrame(orbit_records)
orbit_diagnostics.to_csv(
    TABLE_DIR / "baseline_vza_orbit_diagnostics.csv",
    index=False,
)

display(
    orbit_diagnostics.sort_values(
        "orbit_day_vza_r2",
        ascending=False,
    ).round(3)
)

,class_code,settlement_class,baseline_days,lag16_vza_r,orbit_day_vza_r2,vza_range_degrees
0,11,Very low-density rural,94,0.995,0.997,61.970001
6,30,Urban centre,79,0.996,0.997,64.075996
2,13,Rural cluster,97,0.996,0.997,61.275002
1,12,Low-density rural,102,0.992,0.996,61.728001
3,21,Suburban / peri-urban,106,0.990,0.994,59.784000
5,23,Dense urban cluster,106,0.990,0.993,62.048000
4,22,Semi-dense urban cluster,102,0.985,0.993,59.334000


**Table 6 — Baseline orbital-periodicity diagnostics.** `lag16_vza_r` measures repetition after 16 days; `orbit_day_vza_r2` measures how much baseline VZA variation is structured by orbit day.

**RQ2 relevance:** a strong orbital VZA cycle creates a mechanism for artificial periodicity in radiance. A comparable 16-day radiance pattern should not be interpreted as short-term recovery variation without sensitivity testing.

In [25]:
orbit_profile = (
    orbit_daily.groupby(
        ["class_code", "settlement_class", "orbit_day"],
        observed=True,
    )
    .agg(
        vza_median=("vza_mean", "median"),
        vza_p10=("vza_mean", lambda values: values.quantile(0.10)),
        vza_p90=("vza_mean", lambda values: values.quantile(0.90)),
        dates=("date", "nunique"),
    )
    .reset_index()
)

fig = go.Figure()

for (class_code, settlement_class), group in orbit_profile.groupby(
    ["class_code", "settlement_class"],
    observed=True,
):
    fig.add_trace(
        go.Scatter(
            x=group["orbit_day"],
            y=group["vza_median"],
            mode="lines+markers",
            name=settlement_class,
            line={
                "color": GHSL_CLASS_COLORS.get(class_code, "#666666"),
                "width": 3,
            },
            customdata=np.column_stack(
                [group["vza_p10"], group["vza_p90"], group["dates"]]
            ),
            hovertemplate=(
                "Orbit day: %{x}<br>"
                "Median VZA: %{y:.1f}°<br>"
                "P10-P90: %{customdata[0]:.1f}° to "
                "%{customdata[1]:.1f}°<br>"
                "Dates: %{customdata[2]:.0f}"
                "<extra></extra>"
            ),
        )
    )

fig.update_xaxes(
    title_text="Day within 16-day cycle",
    dtick=1,
)
fig.update_yaxes(title_text="Sensor zenith angle (°)")

fig.update_layout(
    title={
        "text": "Baseline VZA by day within the 16-day VIIRS orbit cycle",
        "x": 0.01,
        "xanchor": "left",
    },
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    font={"family": "Arial", "size": 16, "color": "#111827"},
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.08,
        "xanchor": "right",
        "x": 1,
    },
    margin={"l": 90, "r": 45, "t": 145, "b": 75},
    width=1450,
    height=650,
)

fig.show()
fig.write_html(
    FIGURE_DIR / "04_baseline_vza_orbit_cycle.html",
    include_plotlyjs="cdn",
)

**Figure 4 — VZA across the 16-day orbit cycle.** The figure visualises whether each settlement class follows the same recurring angular sequence and whether any class has narrower or shifted VZA support.

**RQ2 relevance:** shared curves indicate an orbit-driven regional pattern; class-specific differences indicate additional sampling or settlement-location effects.

## 6.1 Daily VZA–radiance association by settlement and product

The daily analysis uses baseline-relative pixel medians, reducing sensitivity to persistent brightness differences. Direct DNB-BRDF, paired gap-filled, and operational gap-filled supports remain separate. `vza_time_spearman` is retained as a temporal-confounding diagnostic.

In [26]:
DAILY_PRODUCTS = {
    "DNB-BRDF": "dnb_relative_median",
    "Gap-filled paired": "gap_paired_relative_median",
    "Gap-filled operational": "gap_all_relative_median",
}

daily_association_records = []

for (class_code, phase_name), group in daily_summary.groupby(
    ["class_code", "phase"],
    observed=True,
):
    for product_name, value_column in DAILY_PRODUCTS.items():
        analysis_data = (
            group[
                [
                    "vza_median",
                    value_column,
                    "observed_pixels",
                    "valid_pct",
                    "relative_day",
                ]
            ]
            .replace([np.inf, -np.inf], np.nan)
            .dropna(subset=["vza_median", value_column])
            .loc[lambda frame: frame["observed_pixels"] >= MIN_DAILY_PIXELS]
        )

        if len(analysis_data) >= 3:
            spearman_rho = analysis_data["vza_median"].corr(
                analysis_data[value_column],
                method="spearman",
            )

            vza_time_spearman = analysis_data["vza_median"].corr(
                analysis_data["relative_day"],
                method="spearman",
            )

            slope_per_degree, intercept = np.polyfit(
                analysis_data["vza_median"],
                analysis_data[value_column],
                deg=1,
            )
        else:
            spearman_rho = np.nan
            vza_time_spearman = np.nan
            slope_per_degree = np.nan
            intercept = np.nan

        daily_association_records.append(
            {
                "class_code": class_code,
                "settlement_class": GHSL_CLASS_LABELS[class_code],
                "phase": phase_name,
                "product": product_name,
                "dates": len(analysis_data),
                "median_valid_pct": analysis_data["valid_pct"].median(),
                "spearman_rho": spearman_rho,
                "vza_time_spearman": vza_time_spearman,
                "linear_change_pp_per_10deg": 1_000 * slope_per_degree,
                "intercept": intercept,
            }
        )

daily_association = pd.DataFrame(daily_association_records)

daily_association.to_csv(
    TABLE_DIR / "daily_vza_radiance_association.csv",
    index=False,
)

display(
    daily_association.loc[
        daily_association["phase"].isin(["Baseline", "Long-term"])
    ].style.format(
        {
            "median_valid_pct": "{:.1f}",
            "spearman_rho": "{:.3f}",
            "vza_time_spearman": "{:.3f}",
            "linear_change_pp_per_10deg": "{:+.2f}",
            "intercept": "{:.3f}",
        },
        na_rep="—",
    )
)

,class_code,settlement_class,phase,product,dates,median_valid_pct,spearman_rho,vza_time_spearman,linear_change_pp_per_10deg,intercept
0,11,Very low-density rural,Baseline,DNB-BRDF,64,67.5,0.339,-0.061,+9.12,-0.413
1,11,Very low-density rural,Baseline,Gap-filled paired,64,67.5,0.339,-0.061,+9.12,-0.413
2,11,Very low-density rural,Baseline,Gap-filled operational,64,67.5,0.358,-0.061,+4.93,-0.184
12,11,Very low-density rural,Long-term,DNB-BRDF,87,64.0,0.297,-0.092,+8.72,-0.661
13,11,Very low-density rural,Long-term,Gap-filled paired,87,64.0,0.297,-0.092,+8.72,-0.661
14,11,Very low-density rural,Long-term,Gap-filled operational,87,64.0,0.215,-0.092,+5.31,-0.464
15,12,Low-density rural,Baseline,DNB-BRDF,79,43.6,0.475,-0.060,+11.23,-0.576
16,12,Low-density rural,Baseline,Gap-filled paired,79,43.6,0.475,-0.060,+11.23,-0.576
17,12,Low-density rural,Baseline,Gap-filled operational,79,43.6,0.267,-0.060,+4.10,-0.141
27,12,Low-density rural,Long-term,DNB-BRDF,108,46.0,0.384,-0.124,+10.04,-0.670


**Table 7 — Daily VZA–radiance association.** The table reports date support, valid-pixel coverage, Spearman correlation, temporal confounding, and the fitted change per 10°.

**RQ2 relevance:** a strong VZA–radiance association is only credible as geometry sensitivity when it is sufficiently supported and not primarily a VZA–time relationship.

## 6.2 Baseline VZA–radiance response curves

Daily values are binned in 10° intervals. Lines show bin medians rather than fitted correction curves. The baseline-only restriction prevents the Haiyan signal from being learned as an angular response.

In [27]:
baseline_daily = (
    daily_summary.loc[
        (daily_summary["phase"] == "Baseline")
        & (daily_summary["observed_pixels"] >= MIN_DAILY_PIXELS)
    ]
    .replace([np.inf, -np.inf], np.nan)
    .copy()
)

baseline_daily["vza_bin"] = pd.cut(
    baseline_daily["vza_median"],
    bins=VZA_BINS,
    include_lowest=True,
    right=False,
)

product_styles = {
    "DNB-BRDF": {
        "column": "dnb_relative_median",
        "color": DNB_COLOR,
        "symbol": "circle",
    },
    "Gap-filled paired": {
        "column": "gap_paired_relative_median",
        "color": GAP_COLOR,
        "symbol": "diamond",
    },
}

fig = make_subplots(
    rows=2,
    cols=4,
    horizontal_spacing=0.075,
    vertical_spacing=0.16,
    subplot_titles=[
        GHSL_CLASS_LABELS[class_code]
        for class_code in GHSL_CLASS_LABELS
    ]
    + [""],
)

for plot_index, (class_code, class_label) in enumerate(
    GHSL_CLASS_LABELS.items()
):
    row = plot_index // 4 + 1
    column = plot_index % 4 + 1

    class_data = baseline_daily.loc[
        baseline_daily["class_code"] == class_code
    ]

    for product_name, style in product_styles.items():
        value_column = style["column"]

        product_data = class_data.dropna(
            subset=["vza_median", value_column]
        )

        binned = (
            product_data
            .groupby("vza_bin", observed=True)
            .agg(
                median_vza=("vza_median", "median"),
                median_anomaly=(value_column, "median"),
                q25=(
                    value_column,
                    lambda values: values.quantile(0.25),
                ),
                q75=(
                    value_column,
                    lambda values: values.quantile(0.75),
                ),
                dates=(value_column, "size"),
            )
            .reset_index()
        )

        fig.add_trace(
            go.Scatter(
                x=product_data["vza_median"],
                y=100 * product_data[value_column],
                mode="markers",
                name=product_name,
                legendgroup=product_name,
                showlegend=plot_index == 0,
                marker={
                    "size": 4,
                    "color": style["color"],
                    "opacity": 0.14,
                },
                hovertemplate=(
                    "Median VZA: %{x:.2f}°<br>"
                    "Baseline-relative anomaly: %{y:.1f}%"
                    "<extra></extra>"
                ),
            ),
            row=row,
            col=column,
        )

        fig.add_trace(
            go.Scatter(
                x=binned["median_vza"],
                y=100 * binned["median_anomaly"],
                mode="lines+markers",
                name=f"{product_name} bin median",
                legendgroup=product_name,
                showlegend=False,
                line={"color": style["color"], "width": 2.5},
                marker={
                    "size": 8,
                    "symbol": style["symbol"],
                },
                error_y={
                    "type": "data",
                    "symmetric": False,
                    "array": 100 * (binned["q75"] - binned["median_anomaly"]),
                    "arrayminus": 100 * (binned["median_anomaly"] - binned["q25"]),
                    "color": style["color"],
                    "thickness": 1.2,
                    "width": 3,
                },
                customdata=binned["dates"],
                hovertemplate=(
                    "Median VZA: %{x:.2f}°<br>"
                    "Median anomaly: %{y:.1f}%<br>"
                    "Dates: %{customdata}"
                    "<extra></extra>"
                ),
            ),
            row=row,
            col=column,
        )

    fig.add_hline(
        y=0,
        line={"color": "#94A3B8", "width": 1, "dash": "dash"},
        row=row,
        col=column,
    )

    fig.update_xaxes(
        title_text="Median VZA (°)" if row == 2 else None,
        range=[0, 70],
        row=row,
        col=column,
    )

    fig.update_yaxes(
        title_text="Anomaly (%)" if column == 1 else None,
        row=row,
        col=column,
    )

fig.update_layout(
    title={
        "text": "Baseline VZA–radiance association across settlement classes",
        "x": 0.01,
        "xanchor": "left",
    },
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    font={"family": "Arial", "size": 15, "color": "#111827"},
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.08,
        "xanchor": "right",
        "x": 1,
    },
    margin={"l": 85, "r": 45, "t": 145, "b": 75},
    width=1600,
    height=930,
)

fig.show()
fig.write_html(
    FIGURE_DIR / "05_baseline_vza_radiance_by_settlement.html",
    include_plotlyjs="cdn",
)

**Figure 5 — Baseline VZA–radiance response.** Flat, overlapping responses indicate limited residual angular sensitivity. Coherent displacement or curvature indicates that identical settlement classes appear systematically brighter or darker at different VZA.

**RQ2 relevance:** the magnitude of this baseline relationship provides a scale against which the Haiyan-related decline should be judged; it does not by itself justify correction.

## 7.1 Stratified pixel sample

Daily settlement means can change when different pixels are observed. A fixed sample of baseline-lit pixels from each atomic GHSL class is therefore followed through time. `MAX_PIXELS_PER_CLASS` controls memory use.

In [28]:
rng = np.random.default_rng(RANDOM_SEED)

sample_y_indices = []
sample_x_indices = []
sample_class_codes = []

for class_code in GHSL_CLASS_LABELS:
    candidate_mask = (
        (ghsl == class_code)
        & signal_mask
    ).values

    y_indices, x_indices = np.where(candidate_mask)
    candidate_count = len(y_indices)

    if candidate_count == 0:
        continue

    sample_count = min(MAX_PIXELS_PER_CLASS, candidate_count)
    selected = rng.choice(
        candidate_count,
        size=sample_count,
        replace=False,
    )

    sample_y_indices.extend(y_indices[selected])
    sample_x_indices.extend(x_indices[selected])
    sample_class_codes.extend([class_code] * sample_count)

sample_y_indices = np.asarray(sample_y_indices, dtype=int)
sample_x_indices = np.asarray(sample_x_indices, dtype=int)
sample_class_codes = np.asarray(sample_class_codes, dtype=int)

if len(sample_class_codes) == 0:
    raise ValueError("No pixels are available for the stratified VZA sample.")

y_selector = xr.DataArray(
    sample_y_indices,
    dims="sample",
)

x_selector = xr.DataArray(
    sample_x_indices,
    dims="sample",
)

sample_cube = xr.Dataset(
    {
        "dnb": dnb.isel(y=y_selector, x=x_selector),
        "gap_filled": gap_filled.isel(y=y_selector, x=x_selector),
        "mqf": mqf.isel(y=y_selector, x=x_selector),
        "vza": sensor_zenith.isel(y=y_selector, x=x_selector),
    }
).compute()

sample_inventory = (
    pd.Series(sample_class_codes)
    .value_counts()
    .sort_index()
    .rename_axis("class_code")
    .reset_index(name="sampled_pixels")
)

sample_inventory["settlement_class"] = sample_inventory[
    "class_code"
].map(GHSL_CLASS_LABELS)

display(sample_inventory)

,class_code,sampled_pixels,settlement_class
0,11,114,Very low-density rural
1,12,150,Low-density rural
2,13,123,Rural cluster
3,21,150,Suburban / peri-urban
4,22,150,Semi-dense urban cluster
5,23,150,Dense urban cluster
6,30,150,Urban centre


**Table 8 — Stratified sample inventory.** The table records the number of fixed pixels sampled from each atomic GHSL class.

**RQ2 relevance:** unequal or very small samples weaken class comparisons; fixed pixels prevent changing spatial support from masquerading as angular response.

In [29]:
sample_dates = pd.DatetimeIndex(sample_cube["date"].values)
number_of_dates = len(sample_dates)
number_of_pixels = len(sample_class_codes)

dnb_values = (
    sample_cube["dnb"]
    .transpose("date", "sample")
    .values
)

gap_values = (
    sample_cube["gap_filled"]
    .transpose("date", "sample")
    .values
)

mqf_values = (
    sample_cube["mqf"]
    .transpose("date", "sample")
    .values
)

vza_values = (
    sample_cube["vza"]
    .transpose("date", "sample")
    .values
)

dnb_qualified_values = np.where(
    np.isin(mqf_values, ACCEPTED_MQF_VALUES),
    dnb_values,
    np.nan,
)

gap_paired_values = np.where(
    np.isfinite(dnb_qualified_values),
    gap_values,
    np.nan,
)

baseline_date_mask = sample_dates < EVENT_DATE

baseline_log_dnb = np.nanmedian(
    np.log1p(dnb_qualified_values[baseline_date_mask]),
    axis=0,
)

baseline_log_gap_paired = np.nanmedian(
    np.log1p(gap_paired_values[baseline_date_mask]),
    axis=0,
)

baseline_log_gap_all = np.nanmedian(
    np.log1p(gap_values[baseline_date_mask]),
    axis=0,
)

sample_df = pd.DataFrame(
    {
        "date": np.repeat(sample_dates, number_of_pixels),
        "sample_id": np.tile(
            np.arange(number_of_pixels),
            number_of_dates,
        ),
        "class_code": np.tile(
            sample_class_codes,
            number_of_dates,
        ),
        "vza": vza_values.reshape(-1),
        "dnb": dnb_qualified_values.reshape(-1),
        "gap_paired": gap_paired_values.reshape(-1),
        "gap_all": gap_values.reshape(-1),
        "dnb_log_anomaly": (
            np.log1p(dnb_qualified_values)
            - baseline_log_dnb[None, :]
        ).reshape(-1),
        "gap_paired_log_anomaly": (
            np.log1p(gap_paired_values)
            - baseline_log_gap_paired[None, :]
        ).reshape(-1),
        "gap_all_log_anomaly": (
            np.log1p(gap_values)
            - baseline_log_gap_all[None, :]
        ).reshape(-1),
    }
)

sample_df["settlement_class"] = sample_df["class_code"].map(
    GHSL_CLASS_LABELS
)

sample_df = assign_phase(sample_df)

print(
    f"Pixel sample: {number_of_pixels:,} pixels × "
    f"{number_of_dates:,} days = {len(sample_df):,} potential records"
)

Pixel sample: 987 pixels × 546 days = 538,902 potential records


## 7.2 Within-pixel VZA slopes

For each sampled pixel, stable phase, and product, a simple VZA slope is fitted to baseline-centred `log1p` radiance. This controls persistent pixel brightness. Pixels must meet the observation-count and VZA-spread rules defined in the settings cell.

In [30]:
PIXEL_PRODUCTS = {
    "DNB-BRDF": "dnb_log_anomaly",
    "Gap-filled paired": "gap_paired_log_anomaly",
    "Gap-filled operational": "gap_all_log_anomaly",
}

pixel_slope_records = []

for phase_name in ["Baseline", "Long-term"]:
    phase_data = sample_df.loc[
        sample_df["phase"] == phase_name
    ]

    for (
        class_code,
        sample_id,
    ), pixel_data in phase_data.groupby(
        ["class_code", "sample_id"],
        observed=True,
    ):
        for product_name, value_column in PIXEL_PRODUCTS.items():
            valid = (
                pixel_data[["vza", value_column]]
                .replace([np.inf, -np.inf], np.nan)
                .dropna()
            )

            if (
                len(valid) < MIN_PIXEL_OBSERVATIONS
                or valid["vza"].std() < MIN_PIXEL_VZA_SD
            ):
                continue

            slope_per_degree, intercept = np.polyfit(
                valid["vza"],
                valid[value_column],
                deg=1,
            )

            spearman_rho = valid["vza"].corr(
                valid[value_column],
                method="spearman",
            )

            pixel_slope_records.append(
                {
                    "phase": phase_name,
                    "class_code": class_code,
                    "settlement_class": GHSL_CLASS_LABELS[class_code],
                    "sample_id": sample_id,
                    "product": product_name,
                    "observations": len(valid),
                    "vza_sd": valid["vza"].std(),
                    "spearman_rho": spearman_rho,
                    "slope_log_per_degree": slope_per_degree,
                    "radiance_change_pct_per_10deg": (
                        100 * np.expm1(10 * slope_per_degree)
                    ),
                    "intercept": intercept,
                }
            )

pixel_slopes = pd.DataFrame(pixel_slope_records)

if pixel_slopes.empty:
    raise ValueError(
        "No sampled pixels satisfy the within-pixel slope requirements."
    )

pixel_slopes.to_csv(
    TABLE_DIR / "pixel_vza_slopes.csv",
    index=False,
)

display(
    pixel_slopes.head().style.format(
        {
            "vza_sd": "{:.2f}",
            "spearman_rho": "{:.3f}",
            "slope_log_per_degree": "{:+.5f}",
            "radiance_change_pct_per_10deg": "{:+.2f}",
            "intercept": "{:.3f}",
        }
    )
)

,phase,class_code,settlement_class,sample_id,product,observations,vza_sd,spearman_rho,slope_log_per_degree,radiance_change_pct_per_10deg,intercept
0,Baseline,11,Very low-density rural,0,DNB-BRDF,41,19.44,0.055,+0.00314,+3.19,0.005
1,Baseline,11,Very low-density rural,0,Gap-filled paired,41,19.44,0.055,+0.00314,+3.19,0.005
2,Baseline,11,Very low-density rural,0,Gap-filled operational,180,19.22,0.006,+0.00002,+0.02,-0.004
3,Baseline,11,Very low-density rural,1,DNB-BRDF,28,16.90,0.246,+0.01011,+10.64,-0.346
4,Baseline,11,Very low-density rural,1,Gap-filled paired,28,16.90,0.246,+0.01011,+10.64,-0.346


**Table 9 — Pixel-level slope preview.** Each row is one supported pixel–phase–product model with observation count, VZA spread, rank correlation, and implied percentage change per 10°.

**RQ2 relevance:** these fits compare the same locations through time and therefore provide stronger geometry evidence than settlement means alone.

In [31]:
def bootstrap_median_ci(values, repetitions, random_seed):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan, np.nan

    local_rng = np.random.default_rng(random_seed)
    bootstrap_indices = local_rng.integers(
        0,
        len(values),
        size=(repetitions, len(values)),
    )
    bootstrap_medians = np.median(
        values[bootstrap_indices],
        axis=1,
    )

    return tuple(
        np.quantile(bootstrap_medians, [0.025, 0.975])
    )


effect_records = []

for (
    phase_name,
    class_code,
    settlement_class,
    product_name,
), group in pixel_slopes.groupby(
    [
        "phase",
        "class_code",
        "settlement_class",
        "product",
    ],
    observed=True,
):
    values = group["radiance_change_pct_per_10deg"].dropna().to_numpy()

    ci_low, ci_high = bootstrap_median_ci(
        values,
        repetitions=BOOTSTRAP_REPETITIONS,
        random_seed=RANDOM_SEED + int(class_code),
    )

    median_change = np.median(values) if len(values) else np.nan
    directional_evidence = (
        np.isfinite(ci_low)
        and np.isfinite(ci_high)
        and ((ci_low > 0) or (ci_high < 0))
    )
    material_change = (
        np.isfinite(median_change)
        and abs(median_change) >= MATERIAL_CHANGE_PCT_PER_10DEG
    )

    effect_records.append(
        {
            "phase": phase_name,
            "class_code": class_code,
            "settlement_class": settlement_class,
            "product": product_name,
            "pixels": len(values),
            "median_observations": group["observations"].median(),
            "median_spearman_rho": group["spearman_rho"].median(),
            "median_change_pct_per_10deg": median_change,
            "q25_change_pct_per_10deg": np.quantile(values, 0.25),
            "q75_change_pct_per_10deg": np.quantile(values, 0.75),
            "bootstrap_ci_low": ci_low,
            "bootstrap_ci_high": ci_high,
            "directional_evidence": directional_evidence,
            "material_change": material_change,
            "geometry_sensitive_screen": (
                directional_evidence and material_change
            ),
        }
    )

effect_summary = pd.DataFrame(effect_records)

effect_summary.to_csv(
    TABLE_DIR / "within_pixel_vza_effect_summary.csv",
    index=False,
)

display(
    effect_summary.style.format(
        {
            "median_observations": "{:.0f}",
            "median_spearman_rho": "{:.3f}",
            "median_change_pct_per_10deg": "{:+.2f}",
            "q25_change_pct_per_10deg": "{:+.2f}",
            "q75_change_pct_per_10deg": "{:+.2f}",
            "bootstrap_ci_low": "{:+.2f}",
            "bootstrap_ci_high": "{:+.2f}",
        },
        na_rep="—",
    )
)

,phase,class_code,settlement_class,product,pixels,median_observations,median_spearman_rho,median_change_pct_per_10deg,q25_change_pct_per_10deg,q75_change_pct_per_10deg,bootstrap_ci_low,bootstrap_ci_high,directional_evidence,material_change,geometry_sensitive_screen
0,Baseline,11,Very low-density rural,DNB-BRDF,114,40,0.211,+4.99,+2.57,+7.86,+4.11,+5.88,True,False,False
1,Baseline,11,Very low-density rural,Gap-filled operational,114,180,0.057,+0.95,+0.49,+1.51,+0.86,+1.14,True,False,False
2,Baseline,11,Very low-density rural,Gap-filled paired,114,40,0.211,+4.99,+2.57,+7.86,+4.11,+5.88,True,False,False
3,Baseline,12,Low-density rural,DNB-BRDF,150,37,0.248,+4.45,+2.66,+7.89,+3.99,+6.01,True,False,False
4,Baseline,12,Low-density rural,Gap-filled operational,150,180,0.063,+0.95,+0.43,+1.52,+0.74,+1.13,True,False,False
5,Baseline,12,Low-density rural,Gap-filled paired,150,37,0.248,+4.45,+2.66,+7.89,+3.99,+6.01,True,False,False
6,Baseline,13,Rural cluster,DNB-BRDF,123,38,0.193,+4.14,+2.29,+6.58,+3.58,+4.88,True,False,False
7,Baseline,13,Rural cluster,Gap-filled operational,123,180,0.050,+0.79,+0.50,+1.20,+0.66,+0.90,True,False,False
8,Baseline,13,Rural cluster,Gap-filled paired,123,38,0.193,+4.14,+2.29,+6.58,+3.58,+4.88,True,False,False
9,Baseline,21,Suburban / peri-urban,DNB-BRDF,150,34,0.289,+4.65,+3.03,+7.05,+4.21,+5.24,True,False,False


**Table 10 — Within-pixel VZA effect summary.** The table reports class-level median effects, interquartile ranges, bootstrap confidence intervals, and the transparent materiality screen.

**RQ2 relevance:** directionally inconsistent or highly dispersed effects argue against a universal correction. A stable effect exceeding the materiality rule warrants threshold and correction sensitivity tests.

## 7.3 Distribution of within-pixel VZA effects

The baseline panel is the primary diagnostic. The long-term panel tests whether direction and magnitude persist after the recovery interval. The shaded band marks effects smaller than `MATERIAL_CHANGE_PCT_PER_10DEG`.

In [32]:
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=[
        "Baseline",
        "Long-term stability check",
    ],
)

product_colors = {
    "DNB-BRDF": DNB_COLOR,
    "Gap-filled paired": GAP_COLOR,
    "Gap-filled operational": "#0EA5E9",
}

for row, phase_name in enumerate(
    ["Baseline", "Long-term"],
    start=1,
):
    phase_data = pixel_slopes.loc[
        pixel_slopes["phase"] == phase_name
    ]

    for product_name, color in product_colors.items():
        product_data = phase_data.loc[
            phase_data["product"] == product_name
        ]

        fig.add_trace(
            go.Box(
                x=product_data["settlement_class"],
                y=product_data["radiance_change_pct_per_10deg"],
                name=product_name,
                legendgroup=product_name,
                showlegend=row == 1,
                marker_color=color,
                boxpoints=False,
                hovertemplate=(
                    "Settlement: %{x}<br>"
                    "Change per 10°: %{y:+.2f}%"
                    "<extra></extra>"
                ),
            ),
            row=row,
            col=1,
        )

    fig.add_hline(
        y=0,
        line={"color": "#64748B", "width": 1.2, "dash": "dash"},
        row=row,
        col=1,
    )

    fig.add_hrect(
        y0=-MATERIAL_CHANGE_PCT_PER_10DEG,
        y1=MATERIAL_CHANGE_PCT_PER_10DEG,
        fillcolor="rgba(148,163,184,0.10)",
        line_width=0,
        layer="below",
        row=row,
        col=1,
    )

fig.update_yaxes(
    title_text="Radiance change per 10° (%)",
    row=1,
    col=1,
)
fig.update_yaxes(
    title_text="Radiance change per 10° (%)",
    row=2,
    col=1,
)
fig.update_xaxes(
    title_text="GHSL settlement class",
    tickangle=-18,
    row=2,
    col=1,
)

fig.update_layout(
    title={
        "text": "Within-pixel VZA sensitivity by settlement class and product",
        "x": 0.01,
        "xanchor": "left",
    },
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    font={"family": "Arial", "size": 16, "color": "#111827"},
    boxmode="group",
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.08,
        "xanchor": "right",
        "x": 1,
    },
    margin={"l": 100, "r": 45, "t": 145, "b": 135},
    width=1500,
    height=900,
)

fig.show()
fig.write_html(
    FIGURE_DIR / "06_within_pixel_vza_effects.html",
    include_plotlyjs="cdn",
)

**Figure 6 — Within-pixel VZA effects.** Narrow, similarly signed boxes indicate a coherent angular response; broad or sign-changing boxes indicate pixel-level heterogeneity.

**RQ2 relevance:** heterogeneous effects favour reporting sensitivity rather than applying one regional correction equation.

## 8. Literature-standard VZA intervals

The sampled pixel-days are classified into near-nadir (0–20°), middle (20–40°), off-nadir (40–60°), and extreme (60–70°) intervals. The baseline window isolates ordinary angular behaviour from the disaster trajectory.

In [33]:
vza_sample = sample_df.loc[
    sample_df["vza"].between(0, VZA_MAX_PHYSICAL)
].copy()

vza_sample["orbit_day"] = (
    (vza_sample["date"] - VZA_ORBIT_EPOCH).dt.days % 16
) + 1

vza_sample["vza_interval"] = pd.cut(
    vza_sample["vza"],
    bins=VZA_INTERVAL_EDGES,
    labels=VZA_INTERVAL_LABELS,
    right=False,
    include_lowest=True,
)

vza_sample["view_category"] = np.select(
    [
        vza_sample["vza"] <= 20,
        vza_sample["vza"] >= 40,
    ],
    [
        "Near-nadir (≤20°)",
        "Off-nadir (≥40°)",
    ],
    default="Intermediate (20-40°)",
)

vza_baseline = vza_sample.loc[
    vza_sample["phase"] == "Baseline"
].copy()

print(
    f"Retained {len(vza_sample):,} sampled pixel-days "
    f"with VZA ≤ {VZA_MAX_PHYSICAL:.0f}°"
)
print(
    "Excluded",
    f"{(sample_df['vza'] > VZA_MAX_PHYSICAL).sum():,}",
    "records above the physical analysis limit",
)

Retained 534,680 sampled pixel-days with VZA ≤ 70°
Excluded 6 records above the physical analysis limit


In [34]:
qualified_vza_baseline = vza_baseline.loc[
    vza_baseline["dnb"].notna()
].copy()

angle_daily = (
    qualified_vza_baseline.groupby(
        [
            "date",
            "class_code",
            "settlement_class",
            "vza_interval",
        ],
        observed=True,
    )
    .agg(
        dnb_median=("dnb", "median"),
        gap_paired_median=("gap_paired", "median"),
        dnb_log_anomaly=("dnb_log_anomaly", "median"),
        gap_log_anomaly=("gap_paired_log_anomaly", "median"),
        dnb_pixels=("dnb", "count"),
        gap_pixels=("gap_paired", "count"),
    )
    .reset_index()
)

angle_daily["dnb_relative_pct"] = 100 * np.expm1(
    angle_daily["dnb_log_anomaly"]
)
angle_daily["gap_relative_pct"] = 100 * np.expm1(
    angle_daily["gap_log_anomaly"]
)

angle_summary = (
    angle_daily.groupby(
        ["class_code", "settlement_class", "vza_interval"],
        observed=True,
    )
    .agg(
        dates=("date", "nunique"),
        dnb_mean=("dnb_median", "mean"),
        dnb_sd=("dnb_median", "std"),
        gap_mean=("gap_paired_median", "mean"),
        gap_sd=("gap_paired_median", "std"),
        dnb_relative_median=("dnb_relative_pct", "median"),
        gap_relative_median=("gap_relative_pct", "median"),
        median_dnb_pixels=("dnb_pixels", "median"),
        median_gap_pixels=("gap_pixels", "median"),
    )
    .reset_index()
)

angle_summary["dnb_cv"] = (
    angle_summary["dnb_sd"] / angle_summary["dnb_mean"]
)
angle_summary["gap_cv"] = (
    angle_summary["gap_sd"] / angle_summary["gap_mean"]
)

angle_summary.to_csv(
    TABLE_DIR / "baseline_radiance_by_vza_interval.csv",
    index=False,
)

display(
    angle_summary[
        [
            "settlement_class",
            "vza_interval",
            "dates",
            "dnb_relative_median",
            "gap_relative_median",
            "dnb_cv",
            "gap_cv",
            "median_dnb_pixels",
            "median_gap_pixels",
        ]
    ].round(3)
)

near_off_daily = (
    qualified_vza_baseline.groupby(
        [
            "date",
            "class_code",
            "settlement_class",
            "view_category",
        ],
        observed=True,
    )
    .agg(
        dnb_log_anomaly=("dnb_log_anomaly", "median"),
        gap_log_anomaly=("gap_paired_log_anomaly", "median"),
    )
    .reset_index()
)

near_off_daily["dnb_relative_pct"] = 100 * np.expm1(
    near_off_daily["dnb_log_anomaly"]
)
near_off_daily["gap_relative_pct"] = 100 * np.expm1(
    near_off_daily["gap_log_anomaly"]
)

near_off_summary = (
    near_off_daily.groupby(
        ["class_code", "settlement_class", "view_category"],
        observed=True,
    )
    .agg(
        dates=("date", "nunique"),
        dnb_relative_pct=("dnb_relative_pct", "median"),
        gap_relative_pct=("gap_relative_pct", "median"),
    )
    .reset_index()
)

near = near_off_summary.loc[
    near_off_summary["view_category"] == "Near-nadir (≤20°)"
].rename(
    columns={
        "dates": "near_dates",
        "dnb_relative_pct": "dnb_near_pct",
        "gap_relative_pct": "gap_near_pct",
    }
)

off = near_off_summary.loc[
    near_off_summary["view_category"] == "Off-nadir (≥40°)"
].rename(
    columns={
        "dates": "off_dates",
        "dnb_relative_pct": "dnb_off_pct",
        "gap_relative_pct": "gap_off_pct",
    }
)

near_off_contrast = near.merge(
    off,
    on=["class_code", "settlement_class"],
    how="inner",
)

near_off_contrast["dnb_off_minus_near_pp"] = (
    near_off_contrast["dnb_off_pct"]
    - near_off_contrast["dnb_near_pct"]
)
near_off_contrast["gap_off_minus_near_pp"] = (
    near_off_contrast["gap_off_pct"]
    - near_off_contrast["gap_near_pct"]
)

near_off_contrast.to_csv(
    TABLE_DIR / "baseline_near_off_nadir_contrast.csv",
    index=False,
)

display(
    near_off_contrast[
        [
            "settlement_class",
            "near_dates",
            "off_dates",
            "dnb_off_minus_near_pp",
            "gap_off_minus_near_pp",
        ]
    ].round(2)
)

,settlement_class,vza_interval,dates,dnb_relative_median,gap_relative_median,dnb_cv,gap_cv,median_dnb_pixels,median_gap_pixels
0,Very low-density rural,0-20° near-nadir,21,-11.812000,-11.812000,0.686,0.686,16.0,16.0
1,Very low-density rural,20-40° middle,35,-2.090000,-2.090000,0.652,0.652,10.0,10.0
2,Very low-density rural,40-60° off-nadir,43,1.846000,1.846000,0.540,0.540,18.0,18.0
3,Very low-density rural,60-70° extreme,21,-1.076000,-1.076000,0.887,0.887,22.0,22.0
4,Low-density rural,0-20° near-nadir,24,-15.522000,-15.522000,0.633,0.633,9.5,9.5
5,Low-density rural,20-40° middle,37,-5.794000,-5.794000,0.508,0.508,21.0,21.0
6,Low-density rural,40-60° off-nadir,44,0.842000,0.842000,0.912,0.912,37.0,37.0
7,Low-density rural,60-70° extreme,20,15.372000,15.372000,0.854,0.854,51.5,51.5
8,Rural cluster,0-20° near-nadir,22,-10.990000,-10.990000,1.197,1.197,10.0,10.0
9,Rural cluster,20-40° middle,39,-7.788000,-7.788000,0.781,0.781,25.0,25.0


,settlement_class,near_dates,off_dates,dnb_off_minus_near_pp,gap_off_minus_near_pp
0,Very low-density rural,21,56,17.07,17.07
1,Low-density rural,24,56,19.43,19.43
2,Rural cluster,22,58,13.76,13.76
3,Suburban / peri-urban,20,57,19.93,19.93
4,Semi-dense urban cluster,18,62,20.26,20.26
5,Dense urban cluster,22,57,15.43,15.43
6,Urban centre,12,45,21.00,21.00


**Table 11 — Radiance by VZA interval.** The table reports date support, baseline-relative radiance, coefficient of variation, and median pixel support for each interval and settlement class.

**Table 12 — Near- versus off-nadir contrast.** The percentage-point difference provides an empirical angular-effect magnitude using a model-free comparison.

**RQ2 relevance:** effects much smaller than the Haiyan decline are unlikely to explain the main shock. Large or morphology-dependent contrasts require explicit geometry sensitivity reporting.

In [35]:
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Direct DNB-BRDF",
        "Paired gap-filled radiance",
    ),
)

for interval in VZA_INTERVAL_LABELS:
    interval_data = angle_daily.loc[
        angle_daily["vza_interval"] == interval
    ]

    fig.add_trace(
        go.Box(
            x=interval_data["settlement_class"],
            y=interval_data["dnb_relative_pct"],
            name=interval,
            legendgroup=interval,
            boxpoints=False,
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Box(
            x=interval_data["settlement_class"],
            y=interval_data["gap_relative_pct"],
            name=interval,
            legendgroup=interval,
            showlegend=False,
            boxpoints=False,
        ),
        row=1,
        col=2,
    )

fig.update_yaxes(
    title_text="Baseline-relative radiance (%)",
    row=1,
    col=1,
)
fig.update_yaxes(
    title_text="Baseline-relative radiance (%)",
    row=1,
    col=2,
)
fig.update_xaxes(tickangle=-20)

fig.update_layout(
    title={
        "text": "Radiance distributions under standard VZA intervals",
        "x": 0.01,
        "xanchor": "left",
    },
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    font={"family": "Arial", "size": 15, "color": "#111827"},
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.10,
        "xanchor": "right",
        "x": 1,
    },
    margin={"l": 90, "r": 45, "t": 145, "b": 150},
    width=1550,
    height=650,
)

fig.show()
fig.write_html(
    FIGURE_DIR / "07_radiance_by_vza_interval.html",
    include_plotlyjs="cdn",
)

**Figure 7 — Radiance distributions by VZA interval.** Overlapping boxes indicate limited practical VZA sensitivity; ordered separation indicates a systematic angular response. Differences between the two panels indicate whether paired gap filling preserves or alters that response.

**RQ2 relevance:** the figure determines whether all valid angles can be retained or whether threshold-based sensitivity must accompany recovery metrics.

## 9. Model-free VZA-threshold sensitivity

The same fixed pixels from `PRIMARY_GROUP` are summarised under all valid VZA values and the thresholds specified in `VZA_THRESHOLDS`. This tests whether the event trajectory changes when high-angle observations are removed without fitting a correction model.

In [36]:
primary_sample = sample_df.loc[
    sample_df["class_code"].isin(PRIMARY_CODES)
].copy()

threshold_daily_frames = []

for threshold in VZA_THRESHOLDS:
    threshold_label = (
        "All valid VZA"
        if threshold is None
        else f"VZA ≤ {threshold}°"
    )

    threshold_data = primary_sample.loc[
        primary_sample["vza"].notna()
        & primary_sample["dnb_log_anomaly"].notna()
    ]

    if threshold is not None:
        threshold_data = threshold_data.loc[
            threshold_data["vza"] <= threshold
        ]

    threshold_daily = (
        threshold_data
        .groupby("date", as_index=False)
        .agg(
            dnb_log_anomaly=("dnb_log_anomaly", "median"),
            gap_paired_log_anomaly=(
                "gap_paired_log_anomaly",
                "median",
            ),
            valid_dnb_pixels=("dnb_log_anomaly", "count"),
        )
    )

    threshold_daily["threshold"] = threshold_label
    threshold_daily["threshold_degrees"] = (
        ALL_VZA_SENTINEL if threshold is None else threshold
    )
    threshold_daily_frames.append(threshold_daily)

threshold_daily = assign_phase(
    pd.concat(threshold_daily_frames, ignore_index=True)
)

for value_column in [
    "dnb_log_anomaly",
    "gap_paired_log_anomaly",
]:
    threshold_daily[f"{value_column}_14d"] = (
        threshold_daily
        .groupby("threshold_degrees", observed=True)[value_column]
        .transform(
            lambda values: values.rolling(
                window=ROLLING_WINDOW,
                center=True,
                min_periods=3,
            ).median()
        )
    )

threshold_summary = (
    threshold_daily
    .groupby(
        ["threshold", "threshold_degrees", "phase"],
        observed=True,
    )
    .agg(
        dates=("date", "size"),
        median_valid_dnb_pixels=("valid_dnb_pixels", "median"),
        median_dnb_log_anomaly=("dnb_log_anomaly", "median"),
        median_gap_paired_log_anomaly=(
            "gap_paired_log_anomaly",
            "median",
        ),
    )
    .reset_index()
    .sort_values(["threshold_degrees", "phase"], ascending=[False, True])
)

threshold_summary["median_dnb_change_pct"] = (
    100 * np.expm1(threshold_summary["median_dnb_log_anomaly"])
)

threshold_summary["median_gap_paired_change_pct"] = (
    100 * np.expm1(
        threshold_summary["median_gap_paired_log_anomaly"]
    )
)

threshold_summary.to_csv(
    TABLE_DIR / f"{OUTPUT_TAG}_vza_threshold_sensitivity.csv",
    index=False,
)

display(
    threshold_summary.style.format(
        {
            "median_valid_dnb_pixels": "{:.0f}",
            "median_dnb_log_anomaly": "{:+.3f}",
            "median_gap_paired_log_anomaly": "{:+.3f}",
            "median_dnb_change_pct": "{:+.1f}",
            "median_gap_paired_change_pct": "{:+.1f}",
        },
        na_rep="—",
    )
)

,threshold,threshold_degrees,phase,dates,median_valid_dnb_pixels,median_dnb_log_anomaly,median_gap_paired_log_anomaly,median_dnb_change_pct,median_gap_paired_change_pct
0,All valid VZA,170,Baseline,96,73,-0.053,-0.053,-5.2,-5.2
1,All valid VZA,170,Shock,28,155,-0.482,-0.482,-38.2,-38.2
2,All valid VZA,170,Early recovery,41,103,-0.357,-0.357,-30.0,-30.0
3,All valid VZA,170,Late recovery,81,156,-0.132,-0.132,-12.3,-12.3
4,All valid VZA,170,Long-term,121,94,-0.116,-0.116,-10.9,-10.9
20,VZA ≤ 50°,50,Baseline,61,54,-0.101,-0.101,-9.6,-9.6
21,VZA ≤ 50°,50,Shock,19,70,-0.470,-0.470,-37.5,-37.5
22,VZA ≤ 50°,50,Early recovery,27,97,-0.363,-0.363,-30.5,-30.5
23,VZA ≤ 50°,50,Late recovery,51,141,-0.175,-0.175,-16.1,-16.1
24,VZA ≤ 50°,50,Long-term,84,99,-0.177,-0.177,-16.2,-16.2


**Table 13 — Phase estimates by VZA threshold.** The table reports phase-specific date support, median retained pixels, and direct and paired gap-filled radiance change.

**RQ2 relevance:** stable estimates indicate geometry-robust recovery. Divergence must be interpreted together with the observation loss caused by stricter thresholds.

In [37]:
threshold_palette = [
    "#111827",
    "#7C3AED",
    "#2563EB",
    "#0D9488",
    "#16A34A",
    "#EA580C",
    "#BE123C",
]

threshold_labels = [
    "All valid VZA"
    if threshold is None
    else f"VZA ≤ {threshold}°"
    for threshold in VZA_THRESHOLDS
]

threshold_colors = {
    label: threshold_palette[index % len(threshold_palette)]
    for index, label in enumerate(threshold_labels)
}

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.10,
    subplot_titles=[
        "Direct DNB-BRDF",
        f"Gap-filled on paired {QUALITY_LABEL} support",
    ],
)

for threshold_label, threshold_data in threshold_daily.groupby(
    "threshold",
    observed=True,
):
    color = threshold_colors.get(threshold_label, "#64748B")

    fig.add_trace(
        go.Scatter(
            x=threshold_data["date"],
            y=100 * np.expm1(
                threshold_data["dnb_log_anomaly_14d"]
            ),
            mode="lines",
            name=threshold_label,
            legendgroup=threshold_label,
            line={"color": color, "width": 2.2},
            customdata=threshold_data["valid_dnb_pixels"],
            hovertemplate=(
                "%{x|%Y-%m-%d}<br>"
                "DNB-BRDF change: %{y:+.1f}%<br>"
                "Sampled valid pixels: %{customdata}"
                "<extra></extra>"
            ),
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=threshold_data["date"],
            y=100 * np.expm1(
                threshold_data["gap_paired_log_anomaly_14d"]
            ),
            mode="lines",
            name=threshold_label,
            legendgroup=threshold_label,
            showlegend=False,
            line={"color": color, "width": 2.2},
            customdata=threshold_data["valid_dnb_pixels"],
            hovertemplate=(
                "%{x|%Y-%m-%d}<br>"
                "Paired gap-filled change: %{y:+.1f}%<br>"
                "Sampled valid pixels: %{customdata}"
                "<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

for row in [1, 2]:
    fig.add_hline(
        y=0,
        line={"color": "#94A3B8", "width": 1, "dash": "dot"},
        row=row,
        col=1,
    )

    fig.add_vline(
        x=EVENT_DATE.to_pydatetime(),
        line={"color": HAIYAN_COLOR, "width": 2, "dash": "dash"},
        row=row,
        col=1,
    )

fig.add_annotation(
    x=EVENT_DATE.to_pydatetime(),
    y=1.04,
    xref="x",
    yref="paper",
    text="Haiyan/Yolanda<br>8 Nov 2013",
    showarrow=False,
    xanchor="left",
    yanchor="bottom",
    font={"size": 13, "color": HAIYAN_COLOR},
    bgcolor="rgba(255,255,255,0.88)",
)

fig.update_yaxes(
    title_text="Baseline-relative change (%)",
    row=1,
    col=1,
)
fig.update_yaxes(
    title_text="Baseline-relative change (%)",
    row=2,
    col=1,
)
fig.update_xaxes(title_text="Date", row=2, col=1)

fig.update_layout(
    title={
        "text": f"{PRIMARY_GROUP} nighttime lights sensitivity to VZA thresholds",
        "x": 0.01,
        "xanchor": "left",
    },
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    font={"family": "Arial", "size": 16, "color": "#111827"},
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.11,
        "xanchor": "right",
        "x": 1,
    },
    margin={"l": 100, "r": 50, "t": 155, "b": 75},
    width=1500,
    height=850,
)

fig.show()
fig.write_html(
    FIGURE_DIR / f"08_{OUTPUT_TAG}_vza_threshold_sensitivity.html",
    include_plotlyjs="cdn",
)

**Figure 8 — Recovery trajectory under alternative VZA thresholds.** Agreement across lines indicates that the inferred shock and recovery shape is not driven by high-angle observations. Divergence indicates geometry sensitivity, but increasingly strict thresholds also reduce support.

**RQ2 relevance:** the preferred treatment is the least restrictive threshold that preserves both stable conclusions and adequate observability.

In [38]:
stability_records = []

for product_name, anomaly_column in {
    "DNB-BRDF": "dnb_log_anomaly",
    "Gap-filled paired": "gap_paired_log_anomaly",
}.items():
    for threshold in [None] + STABILITY_THRESHOLDS:
        threshold_code = (
            ALL_VZA_SENTINEL
            if threshold is None
            else threshold
        )
        threshold_label = (
            "All valid VZA"
            if threshold is None
            else f"VZA ≤ {threshold}°"
        )

        threshold_data = threshold_daily.loc[
            threshold_daily["threshold_degrees"] == threshold_code
        ]

        for phase_name in PHASE_ORDER:
            phase_values = threshold_data.loc[
                threshold_data["phase"] == phase_name,
                anomaly_column,
            ].dropna()

            stability_records.append(
                {
                    "product": product_name,
                    "threshold_degrees": threshold_code,
                    "threshold_label": threshold_label,
                    "phase": phase_name,
                    "dates": len(phase_values),
                    "median_change_pct": (
                        100 * np.expm1(phase_values.median())
                        if len(phase_values)
                        else np.nan
                    ),
                }
            )

vza_metric_stability = pd.DataFrame(stability_records)

all_angle_reference = (
    vza_metric_stability.loc[
        vza_metric_stability["threshold_degrees"]
        == ALL_VZA_SENTINEL,
        ["product", "phase", "median_change_pct"],
    ]
    .rename(
        columns={
            "median_change_pct": "all_angle_change_pct"
        }
    )
)

vza_metric_stability = vza_metric_stability.merge(
    all_angle_reference,
    on=["product", "phase"],
    how="left",
)

vza_metric_stability["difference_from_all_angles_pp"] = (
    vza_metric_stability["median_change_pct"]
    - vza_metric_stability["all_angle_change_pct"]
)

display(
    vza_metric_stability.sort_values(
        ["product", "phase", "threshold_degrees"],
        ascending=[True, True, False],
    ).round(2)
)

vza_stability_decision = (
    vza_metric_stability.loc[
        vza_metric_stability["threshold_degrees"].isin(
            STABILITY_THRESHOLDS
        )
    ]
    .groupby(["product", "threshold_label"], observed=True)
    .agg(
        minimum_phase_dates=("dates", "min"),
        maximum_absolute_difference_pp=(
            "difference_from_all_angles_pp",
            lambda values: values.abs().max(),
        ),
    )
    .reset_index()
)

vza_stability_decision["decision"] = np.select(
    [
        vza_stability_decision["minimum_phase_dates"]
        < MIN_PHASE_DATES_FOR_STABILITY,
        vza_stability_decision["maximum_absolute_difference_pp"]
        >= GEOMETRY_SENSITIVITY_PP,
    ],
    [
        "Observation-limited",
        "Geometry-sensitive",
    ],
    default="Geometry-robust",
)

vza_metric_stability.to_csv(
    TABLE_DIR / f"{OUTPUT_TAG}_phase_vza_stability.csv",
    index=False,
)
vza_stability_decision.to_csv(
    TABLE_DIR / f"{OUTPUT_TAG}_vza_stability_decision.csv",
    index=False,
)

display(vza_stability_decision.round(2))

,product,threshold_degrees,threshold_label,phase,dates,median_change_pct,all_angle_change_pct,difference_from_all_angles_pp
0,DNB-BRDF,170,All valid VZA,Baseline,96,-5.150000,-5.150000,0.00
5,DNB-BRDF,40,VZA ≤ 40°,Baseline,48,-11.730000,-5.150000,-6.58
10,DNB-BRDF,30,VZA ≤ 30°,Baseline,37,-13.140000,-5.150000,-7.99
2,DNB-BRDF,170,All valid VZA,Early recovery,41,-30.010000,-30.010000,0.00
7,DNB-BRDF,40,VZA ≤ 40°,Early recovery,20,-32.930000,-30.010000,-2.92
12,DNB-BRDF,30,VZA ≤ 30°,Early recovery,15,-30.500000,-30.010000,-0.49
3,DNB-BRDF,170,All valid VZA,Late recovery,81,-12.350000,-12.350000,0.00
8,DNB-BRDF,40,VZA ≤ 40°,Late recovery,38,-16.559999,-12.350000,-4.21
13,DNB-BRDF,30,VZA ≤ 30°,Late recovery,28,-16.670000,-12.350000,-4.33
4,DNB-BRDF,170,All valid VZA,Long-term,121,-10.940000,-10.940000,0.00


,product,threshold_label,minimum_phase_dates,maximum_absolute_difference_pp,decision
0,DNB-BRDF,VZA ≤ 30°,11,7.99,Geometry-sensitive
1,DNB-BRDF,VZA ≤ 40°,15,7.80,Geometry-sensitive
2,Gap-filled paired,VZA ≤ 30°,11,7.99,Geometry-sensitive
3,Gap-filled paired,VZA ≤ 40°,15,7.80,Geometry-sensitive


**Table 14 — Phase-level threshold stability.** The detailed table reports phase estimates and their difference from the all-angle reference.

**Table 15 — Threshold decision.** The summary classifies each product and threshold using the explicit minimum-date and 5-percentage-point rules.

**RQ2 relevance:** `Geometry-robust` supports retaining all selected-quality observations; `Geometry-sensitive` requires sensitivity ranges; `Observation-limited` means the stricter subset cannot resolve recovery.

## 10. Quadratic response and nadir-equivalent sensitivity

Quadratic fits follow the angular-response literature but are restricted to the pre-Haiyan baseline. Each pixel must meet the observation-count, VZA-span, and maximum-angle support requirements set at the start.

In [39]:
QUADRATIC_PRODUCTS = {
    "DNB-BRDF": "dnb",
    "Gap-filled paired": "gap_paired",
}

quadratic_records = []

for (
    class_code,
    settlement_class,
    sample_id,
), pixel_data in vza_baseline.groupby(
    ["class_code", "settlement_class", "sample_id"],
    observed=True,
):
    for product_name, radiance_column in QUADRATIC_PRODUCTS.items():
        valid = pixel_data[
            ["vza", radiance_column]
        ].dropna().copy()

        if len(valid) < MIN_QUADRATIC_OBSERVATIONS:
            continue

        q1 = valid[radiance_column].quantile(0.25)
        q3 = valid[radiance_column].quantile(0.75)
        iqr = q3 - q1

        if np.isfinite(iqr) and iqr > 0:
            valid = valid.loc[
                valid[radiance_column].between(
                    q1 - IQR_MULTIPLIER * iqr,
                    q3 + IQR_MULTIPLIER * iqr,
                )
            ]

        if len(valid) < MIN_QUADRATIC_OBSERVATIONS:
            continue

        vza = valid["vza"].to_numpy(dtype=float)
        radiance = valid[radiance_column].to_numpy(dtype=float)

        if (
            np.ptp(vza) < MIN_QUADRATIC_VZA_SPAN
            or np.nanmax(vza) < MIN_REQUIRED_MAX_VZA
        ):
            continue

        quadratic_coef = np.polyfit(vza, radiance, 2)
        linear_coef = np.polyfit(vza, radiance, 1)

        quadratic_prediction = np.polyval(quadratic_coef, vza)
        linear_prediction = np.polyval(linear_coef, vza)

        total_ss = np.square(radiance - radiance.mean()).sum()

        if total_ss <= 0:
            continue

        quadratic_r2 = 1 - (
            np.square(radiance - quadratic_prediction).sum()
            / total_ss
        )
        linear_r2 = 1 - (
            np.square(radiance - linear_prediction).sum()
            / total_ss
        )

        a, b, c = quadratic_coef
        predicted_30 = a * 30**2 + b * 30 + c
        predicted_60 = a * 60**2 + b * 60 + c

        if (
            not np.isfinite(c)
            or c <= 0
            or predicted_30 <= 0
            or predicted_60 <= 0
        ):
            continue

        vertex = -b / (2 * a) if abs(a) > 1e-12 else np.nan
        slope_at_0 = b
        slope_at_60 = 120 * a + b

        if a > 0 and 0 < vertex < MAX_MODEL_VZA:
            response_shape = "U-shaped"
        elif a < 0 and 0 < vertex < MAX_MODEL_VZA:
            response_shape = "Inverted-U"
        elif slope_at_0 >= 0 and slope_at_60 >= 0:
            response_shape = "Positive"
        elif slope_at_0 <= 0 and slope_at_60 <= 0:
            response_shape = "Negative"
        else:
            response_shape = "Mixed"

        quadratic_records.append(
            {
                "class_code": class_code,
                "settlement_class": settlement_class,
                "sample_id": sample_id,
                "product": product_name,
                "observations": len(valid),
                "vza_min": vza.min(),
                "vza_max": vza.max(),
                "vza_span": np.ptp(vza),
                "a": a,
                "b": b,
                "c": c,
                "a_over_c": a / c,
                "b_over_c": b / c,
                "vertex_degrees": vertex,
                "linear_r2": linear_r2,
                "quadratic_r2": quadratic_r2,
                "delta_r2": quadratic_r2 - linear_r2,
                "bias_30_pct": 100 * (predicted_30 / c - 1),
                "bias_60_pct": 100 * (predicted_60 / c - 1),
                "response_shape": response_shape,
            }
        )

quadratic_fits = pd.DataFrame(quadratic_records)
print(f"Usable quadratic models: {len(quadratic_fits):,}")

if quadratic_fits.empty:
    quadratic_summary = pd.DataFrame()
    shape_share = pd.DataFrame()
    print(
        "No pixels met the quadratic support requirements. "
        "Inspect observation counts before relaxing the criteria."
    )
else:
    quadratic_summary = (
        quadratic_fits.groupby(
            ["class_code", "settlement_class", "product"],
            observed=True,
        )
        .agg(
            fitted_pixels=("sample_id", "nunique"),
            median_observations=("observations", "median"),
            median_vza_span=("vza_span", "median"),
            median_linear_r2=("linear_r2", "median"),
            median_quadratic_r2=("quadratic_r2", "median"),
            median_delta_r2=("delta_r2", "median"),
            median_bias_30_pct=("bias_30_pct", "median"),
            median_bias_60_pct=("bias_60_pct", "median"),
        )
        .reset_index()
    )

    shape_share = (
        pd.crosstab(
            [
                quadratic_fits["class_code"],
                quadratic_fits["settlement_class"],
                quadratic_fits["product"],
            ],
            quadratic_fits["response_shape"],
            normalize="index",
        )
        .mul(100)
        .round(1)
        .reset_index()
    )

    quadratic_fits.to_csv(
        TABLE_DIR / "pixel_quadratic_vza_models.csv",
        index=False,
    )
    quadratic_summary.to_csv(
        TABLE_DIR / "quadratic_vza_model_summary.csv",
        index=False,
    )
    shape_share.to_csv(
        TABLE_DIR / "quadratic_response_shape_share.csv",
        index=False,
    )

    display(quadratic_summary.round(3))
    display(shape_share)

Usable quadratic models: 1,860


,class_code,settlement_class,product,fitted_pixels,median_observations,median_vza_span,median_linear_r2,median_quadratic_r2,median_delta_r2,median_bias_30_pct,median_bias_60_pct
0,11,Very low-density rural,DNB-BRDF,111,37.0,61.360,0.030,0.065,0.009,22.811,44.421
1,11,Very low-density rural,Gap-filled paired,111,37.0,61.360,0.030,0.065,0.009,22.811,44.421
2,12,Low-density rural,DNB-BRDF,141,35.0,60.770,0.049,0.062,0.012,50.235,70.257
3,12,Low-density rural,Gap-filled paired,141,35.0,60.770,0.049,0.062,0.012,50.235,70.257
4,13,Rural cluster,DNB-BRDF,116,36.0,61.200,0.043,0.067,0.013,12.345,56.671
5,13,Rural cluster,Gap-filled paired,116,36.0,61.200,0.043,0.067,0.013,12.345,56.671
6,21,Suburban / peri-urban,DNB-BRDF,145,33.0,60.410,0.068,0.091,0.011,41.569,98.692
7,21,Suburban / peri-urban,Gap-filled paired,145,33.0,60.410,0.068,0.091,0.011,41.569,98.692
8,22,Semi-dense urban cluster,DNB-BRDF,137,34.0,61.210,0.100,0.118,0.011,39.693,97.620
9,22,Semi-dense urban cluster,Gap-filled paired,137,34.0,61.210,0.100,0.118,0.011,39.693,97.620


response_shape,class_code,settlement_class,product,Inverted-U,Negative,Positive,U-shaped
0,11,Very low-density rural,DNB-BRDF,27.0,5.4,28.8,38.7
1,11,Very low-density rural,Gap-filled paired,27.0,5.4,28.8,38.7
2,12,Low-density rural,DNB-BRDF,30.5,0.0,29.8,39.7
3,12,Low-density rural,Gap-filled paired,30.5,0.0,29.8,39.7
4,13,Rural cluster,DNB-BRDF,25.0,0.9,25.0,49.1
5,13,Rural cluster,Gap-filled paired,25.0,0.9,25.0,49.1
6,21,Suburban / peri-urban,DNB-BRDF,25.5,1.4,35.2,37.9
7,21,Suburban / peri-urban,Gap-filled paired,25.5,1.4,35.2,37.9
8,22,Semi-dense urban cluster,DNB-BRDF,17.5,0.7,38.7,43.1
9,22,Semi-dense urban cluster,Gap-filled paired,17.5,0.7,38.7,43.1


**Table 16 — Quadratic model summary.** `median_delta_r2` tests whether curvature improves on a straight line; `median_bias_30_pct` and `median_bias_60_pct` estimate the supported difference from nadir.

**Table 17 — Response-shape distribution.** Mixed positive, negative, U-shaped, and inverted-U responses indicate that one correction cannot represent all pixels.

**RQ2 relevance:** correction is defensible only when response form and magnitude are coherent and supported; heterogeneity favours threshold sensitivity.

In [40]:
if quadratic_fits.empty:
    quadratic_curves = pd.DataFrame()
    print("Quadratic response curves were not produced.")
else:
    curve_records = []

    for (
        class_code,
        settlement_class,
        product_name,
    ), group in quadratic_fits.groupby(
        ["class_code", "settlement_class", "product"],
        observed=True,
    ):
        for angle in np.arange(0, MAX_MODEL_VZA + 1, 2):
            relative_response = 100 * (
                (
                    group["a"] * angle**2
                    + group["b"] * angle
                    + group["c"]
                )
                / group["c"]
                - 1
            )

            curve_records.append(
                {
                    "class_code": class_code,
                    "settlement_class": settlement_class,
                    "product": product_name,
                    "vza": angle,
                    "median_relative_pct": relative_response.median(),
                    "p25_relative_pct": relative_response.quantile(0.25),
                    "p75_relative_pct": relative_response.quantile(0.75),
                    "pixels": group["sample_id"].nunique(),
                }
            )

    quadratic_curves = pd.DataFrame(curve_records)

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=(
            "Direct DNB-BRDF",
            "Paired gap-filled",
        ),
        shared_yaxes=True,
    )

    for column_number, product_name in enumerate(
        ["DNB-BRDF", "Gap-filled paired"],
        start=1,
    ):
        product_data = quadratic_curves.loc[
            quadratic_curves["product"] == product_name
        ]

        for (
            class_code,
            settlement_class,
        ), group in product_data.groupby(
            ["class_code", "settlement_class"],
            observed=True,
        ):
            fig.add_trace(
                go.Scatter(
                    x=group["vza"],
                    y=group["median_relative_pct"],
                    mode="lines",
                    name=settlement_class,
                    legendgroup=settlement_class,
                    showlegend=column_number == 1,
                    line={
                        "color": GHSL_CLASS_COLORS.get(
                            class_code,
                            "#666666",
                        ),
                        "width": 3,
                    },
                    customdata=np.column_stack(
                        [
                            group["p25_relative_pct"],
                            group["p75_relative_pct"],
                            group["pixels"],
                        ]
                    ),
                    hovertemplate=(
                        "VZA: %{x:.0f}°<br>"
                        "Median response: %{y:.1f}%<br>"
                        "IQR: %{customdata[0]:.1f}% to "
                        "%{customdata[1]:.1f}%<br>"
                        "Pixels: %{customdata[2]:.0f}"
                        "<extra></extra>"
                    ),
                ),
                row=1,
                col=column_number,
            )

        fig.add_hline(
            y=0,
            line={"color": "#555555", "width": 1, "dash": "dot"},
            row=1,
            col=column_number,
        )

    fig.update_xaxes(title_text="Sensor zenith angle (°)")
    fig.update_yaxes(
        title_text="Expected difference from nadir (%)",
        row=1,
        col=1,
    )

    fig.update_layout(
        title={
            "text": "Estimated VZA response by GHSL settlement class",
            "x": 0.01,
            "xanchor": "left",
        },
        template="plotly_white",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="white",
        font={"family": "Arial", "size": 15, "color": "#111827"},
        legend={
            "orientation": "h",
            "yanchor": "bottom",
            "y": 1.10,
            "xanchor": "right",
            "x": 1,
        },
        margin={"l": 90, "r": 45, "t": 145, "b": 80},
        width=1500,
        height=650,
    )

    fig.show()
    fig.write_html(
        FIGURE_DIR / "09_quadratic_vza_response.html",
        include_plotlyjs="cdn",
    )

**Figure 9 — Estimated VZA response by GHSL class.** The median curve shows the typical expected difference from nadir; the IQR represents pixel-level heterogeneity.

**RQ2 relevance:** narrow, coherent curves may support class-specific correction. Wide or crossing curves indicate that correction would add model uncertainty rather than remove it.

### 10.1 Nadir-equivalent sensitivity trajectory

The correction uses only pixels with supported baseline quadratic fits. Raw and corrected series use exactly the same pixel-days, so differences reflect the fitted VZA transformation rather than changing support.

In [41]:
selected_quadratic_fits = quadratic_fits.loc[
    quadratic_fits["class_code"].isin(PRIMARY_CODES)
].copy()

if selected_quadratic_fits.empty:
    vza_corrected = pd.DataFrame()
    correction_daily = pd.DataFrame()
    correction_phase_summary = pd.DataFrame()
    factor_diagnostics = pd.DataFrame()
    print("Nadir-equivalent sensitivity was not produced.")
else:
    vza_corrected = vza_sample.loc[
        vza_sample["class_code"].isin(PRIMARY_CODES)
    ].copy()

    correction_products = [
        (
            "DNB-BRDF",
            "dnb",
            "dnb_nadir_equivalent",
            "dnb_vza_factor",
        ),
        (
            "Gap-filled paired",
            "gap_paired",
            "gap_nadir_equivalent",
            "gap_vza_factor",
        ),
    ]

    for (
        product_name,
        raw_column,
        corrected_column,
        factor_column,
    ) in correction_products:
        coefficients = (
            selected_quadratic_fits.loc[
                selected_quadratic_fits["product"] == product_name,
                ["sample_id", "a", "b", "c"],
            ]
            .drop_duplicates("sample_id")
            .set_index("sample_id")
        )

        a = vza_corrected["sample_id"].map(coefficients["a"])
        b = vza_corrected["sample_id"].map(coefficients["b"])
        c = vza_corrected["sample_id"].map(coefficients["c"])

        expected_at_observed_vza = (
            a * np.square(vza_corrected["vza"])
            + b * vza_corrected["vza"]
            + c
        )
        factor = c / expected_at_observed_vza

        valid_factor = (
            factor.notna()
            & np.isfinite(factor)
            & (factor > 0)
            & expected_at_observed_vza.gt(0)
        )

        vza_corrected[factor_column] = factor.where(valid_factor)
        vza_corrected[corrected_column] = (
            vza_corrected[raw_column]
            * vza_corrected[factor_column]
        )

    vza_corrected["dnb_raw_supported"] = vza_corrected["dnb"].where(
        vza_corrected["dnb_nadir_equivalent"].notna()
    )
    vza_corrected["gap_raw_supported"] = vza_corrected[
        "gap_paired"
    ].where(
        vza_corrected["gap_nadir_equivalent"].notna()
    )

    factor_diagnostics = pd.DataFrame(
        {
            "DNB factor": vza_corrected["dnb_vza_factor"].describe(
                percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
            ),
            "Gap factor": vza_corrected["gap_vza_factor"].describe(
                percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
            ),
        }
    )
    display(factor_diagnostics.round(3))

    correction_daily = (
        vza_corrected.groupby("date")
        .agg(
            dnb_raw=("dnb_raw_supported", "median"),
            dnb_corrected=("dnb_nadir_equivalent", "median"),
            gap_raw=("gap_raw_supported", "median"),
            gap_corrected=("gap_nadir_equivalent", "median"),
            dnb_pixels=("dnb_nadir_equivalent", "count"),
            gap_pixels=("gap_nadir_equivalent", "count"),
        )
        .reset_index()
        .sort_values("date")
    )

    correction_daily["date"] = pd.to_datetime(
        correction_daily["date"]
    )
    correction_daily = assign_phase(correction_daily)

    correction_series = [
        "dnb_raw",
        "dnb_corrected",
        "gap_raw",
        "gap_corrected",
    ]

    for column in correction_series:
        baseline_median = correction_daily.loc[
            correction_daily["date"] < EVENT_DATE,
            column,
        ].median()

        correction_daily[f"{column}_pct"] = 100 * (
            correction_daily[column] / baseline_median - 1
        )
        correction_daily[f"{column}_rolling"] = (
            correction_daily[f"{column}_pct"]
            .rolling(
                ROLLING_WINDOW,
                min_periods=max(3, ROLLING_WINDOW // 3),
                center=True,
            )
            .median()
        )

    correction_phase_summary = (
        correction_daily.groupby("phase", observed=True)
        .agg(
            dates=("date", "nunique"),
            dnb_raw_pct=("dnb_raw_pct", "median"),
            dnb_corrected_pct=("dnb_corrected_pct", "median"),
            gap_raw_pct=("gap_raw_pct", "median"),
            gap_corrected_pct=("gap_corrected_pct", "median"),
            median_dnb_pixels=("dnb_pixels", "median"),
            median_gap_pixels=("gap_pixels", "median"),
        )
        .reset_index()
    )

    correction_phase_summary["dnb_correction_effect_pp"] = (
        correction_phase_summary["dnb_corrected_pct"]
        - correction_phase_summary["dnb_raw_pct"]
    )
    correction_phase_summary["gap_correction_effect_pp"] = (
        correction_phase_summary["gap_corrected_pct"]
        - correction_phase_summary["gap_raw_pct"]
    )

    correction_phase_summary.to_csv(
        TABLE_DIR / f"{OUTPUT_TAG}_raw_corrected_phase_summary.csv",
        index=False,
    )
    display(correction_phase_summary.round(2))

    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        subplot_titles=(
            "Direct DNB-BRDF",
            "Paired gap-filled",
        ),
        vertical_spacing=0.10,
    )

    fig.add_trace(
        go.Scatter(
            x=correction_daily["date"],
            y=correction_daily["dnb_raw_rolling"],
            name="DNB raw",
            line={"color": DNB_COLOR, "width": 3},
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=correction_daily["date"],
            y=correction_daily["dnb_corrected_rolling"],
            name="DNB nadir-equivalent",
            line={"color": "#7B2CBF", "width": 3},
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=correction_daily["date"],
            y=correction_daily["gap_raw_rolling"],
            name="Gap paired raw",
            line={"color": GAP_COLOR, "width": 3},
        ),
        row=2,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=correction_daily["date"],
            y=correction_daily["gap_corrected_rolling"],
            name="Gap paired nadir-equivalent",
            line={"color": "#00897B", "width": 3},
        ),
        row=2,
        col=1,
    )

    for row_number in [1, 2]:
        fig.add_hline(
            y=0,
            line={"color": "#555555", "width": 1, "dash": "dot"},
            row=row_number,
            col=1,
        )
        fig.add_vline(
            x=EVENT_DATE.to_pydatetime(),
            line={
                "color": HAIYAN_COLOR,
                "width": 2,
                "dash": "dash",
            },
            row=row_number,
            col=1,
        )

    fig.add_annotation(
        x=EVENT_DATE.to_pydatetime(),
        y=1.04,
        xref="x",
        yref="paper",
        text="Haiyan/Yolanda<br>8 Nov 2013",
        showarrow=False,
        xanchor="left",
        yanchor="bottom",
        font={"size": 13, "color": HAIYAN_COLOR},
        bgcolor="rgba(255,255,255,0.88)",
    )

    fig.update_yaxes(
        title_text="Baseline-relative radiance (%)",
        row=1,
        col=1,
    )
    fig.update_yaxes(
        title_text="Baseline-relative radiance (%)",
        row=2,
        col=1,
    )
    fig.update_xaxes(title_text="Date", row=2, col=1)

    fig.update_layout(
        title={
            "text": (
                f"{PRIMARY_GROUP}: raw versus nadir-equivalent "
                f"trajectories"
            ),
            "x": 0.01,
            "xanchor": "left",
        },
        template="plotly_white",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="white",
        font={"family": "Arial", "size": 15, "color": "#111827"},
        legend={
            "orientation": "h",
            "yanchor": "bottom",
            "y": 1.08,
            "xanchor": "right",
            "x": 1,
        },
        margin={"l": 100, "r": 50, "t": 155, "b": 75},
        width=1500,
        height=850,
    )

    fig.show()
    fig.write_html(
        FIGURE_DIR / f"10_{OUTPUT_TAG}_raw_corrected_trajectory.html",
        include_plotlyjs="cdn",
    )

,DNB factor,Gap factor
count,151678.000,151678.000
mean,0.822,0.822
std,0.454,0.454
min,0.013,0.013
1%,0.039,0.039
5%,0.176,0.176
50%,0.808,0.808
95%,1.585,1.585
99%,2.162,2.162
max,4.129,4.129


,phase,dates,dnb_raw_pct,dnb_corrected_pct,gap_raw_pct,gap_corrected_pct,median_dnb_pixels,median_gap_pixels,dnb_correction_effect_pp,gap_correction_effect_pp
0,Baseline,180,0.000000,0.00,0.000000,0.00,2.0,2.0,-0.00,-0.00
1,Shock,31,-78.180000,-78.62,-78.180000,-78.62,120.0,120.0,-0.44,-0.44
2,Early recovery,60,-52.150002,-50.30,-52.150002,-50.30,18.0,18.0,1.84,1.84
3,Late recovery,90,-14.550000,-8.00,-14.550000,-8.00,132.5,132.5,6.55,6.55
4,Long-term,181,-4.140000,-0.51,-4.140000,-0.51,25.0,25.0,3.63,3.63


**Table 18 — Correction-factor diagnostics.** Percentiles reveal whether the fitted transformation contains extreme or unstable factors.

**Table 19 — Raw versus corrected phase estimates.** Correction effects are reported in percentage points using identical supported pixel-days.

**Figure 10 — Raw versus nadir-equivalent trajectories.** Similar shock magnitude, minimum timing, and recovery shape indicate geometry-robust conclusions. Material changes indicate geometry sensitivity.

**RQ2 relevance:** the corrected trajectory remains a sensitivity product; it should not replace direct DNB-BRDF solely because it is smoother.

### 10.2 Does correction reduce the 16-day radiance signal?

The baseline period is used to measure lag-16 correlation and orbit-day \(R^2\) before and after correction. Reduction supports the correction's intended geometric function; it does not alone prove that the corrected disaster trajectory is more accurate.

In [42]:
if correction_daily.empty:
    correction_periodicity = pd.DataFrame()
    periodicity_comparison = pd.DataFrame()
    print("Correction periodicity was not evaluated.")
else:
    correction_daily["orbit_day"] = (
        (correction_daily["date"] - VZA_ORBIT_EPOCH).dt.days % 16
    ) + 1

    periodicity_records = []

    for column in [
        "dnb_raw",
        "dnb_corrected",
        "gap_raw",
        "gap_corrected",
    ]:
        baseline_data = correction_daily.loc[
            correction_daily["phase"] == "Baseline",
            ["date", "orbit_day", column],
        ].dropna().copy()

        if len(baseline_data) < 20:
            continue

        complete_series = (
            baseline_data.set_index("date")[column]
            .reindex(
                pd.date_range(
                    baseline_data["date"].min(),
                    baseline_data["date"].max(),
                    freq="D",
                )
            )
        )
        lag16_r = complete_series.corr(complete_series.shift(16))

        orbit_means = baseline_data.groupby("orbit_day")[column].mean()
        fitted = baseline_data["orbit_day"].map(orbit_means)

        total_ss = np.square(
            baseline_data[column] - baseline_data[column].mean()
        ).sum()
        residual_ss = np.square(
            baseline_data[column] - fitted
        ).sum()

        periodicity_records.append(
            {
                "series": column,
                "baseline_days": len(baseline_data),
                "lag16_r": lag16_r,
                "orbit_day_r2": (
                    1 - residual_ss / total_ss
                    if total_ss > 0
                    else np.nan
                ),
            }
        )

    correction_periodicity = pd.DataFrame(periodicity_records)
    display(correction_periodicity.round(3))

    periodicity_lookup = correction_periodicity.set_index("series")
    comparison_records = []

    for product_name, raw_name, corrected_name in [
        ("DNB-BRDF", "dnb_raw", "dnb_corrected"),
        ("Gap-filled paired", "gap_raw", "gap_corrected"),
    ]:
        if {
            raw_name,
            corrected_name,
        }.issubset(periodicity_lookup.index):
            comparison_records.append(
                {
                    "product": product_name,
                    "raw_orbit_r2": periodicity_lookup.loc[
                        raw_name,
                        "orbit_day_r2",
                    ],
                    "corrected_orbit_r2": periodicity_lookup.loc[
                        corrected_name,
                        "orbit_day_r2",
                    ],
                }
            )

    periodicity_comparison = pd.DataFrame(comparison_records)

    if not periodicity_comparison.empty:
        periodicity_comparison["orbit_r2_reduction_pct"] = np.where(
            periodicity_comparison["raw_orbit_r2"] > 0,
            100
            * (
                1
                - periodicity_comparison["corrected_orbit_r2"]
                / periodicity_comparison["raw_orbit_r2"]
            ),
            np.nan,
        )

        correction_periodicity.to_csv(
            TABLE_DIR / f"{OUTPUT_TAG}_correction_periodicity.csv",
            index=False,
        )
        periodicity_comparison.to_csv(
            TABLE_DIR / f"{OUTPUT_TAG}_periodicity_reduction.csv",
            index=False,
        )

        display(periodicity_comparison.round(2))

,series,baseline_days,lag16_r,orbit_day_r2
0,dnb_raw,96,0.212,0.306
1,dnb_corrected,96,-0.098,0.152
2,gap_raw,96,0.212,0.306
3,gap_corrected,96,-0.098,0.152


,product,raw_orbit_r2,corrected_orbit_r2,orbit_r2_reduction_pct
0,DNB-BRDF,0.31,0.15,50.23
1,Gap-filled paired,0.31,0.15,50.23


**Table 20 — Radiance periodicity before and after correction.** The table reports baseline days, lag-16 correlation, and orbit-day \(R^2\) for each raw and corrected series.

**Table 21 — Reduction in orbit-structured variability.** Positive reduction indicates that correction removed some orbit-day structure; zero or negative reduction indicates no improvement.

**RQ2 relevance:** correction is useful only if it reduces angular periodicity while retaining plausible factors, adequate support, and the substantive disaster signal.

## 11. Synthesis and RQ2 decision

### 11.1 Atomic-class VZA evidence

The table combines baseline VZA distributions, daily settlement-level associations, within-pixel median effects, bootstrap uncertainty, and the exploratory geometry-sensitivity screen.

In [43]:
baseline_effects = effect_summary.loc[
    effect_summary["phase"] == "Baseline",
    [
        "class_code",
        "settlement_class",
        "product",
        "pixels",
        "median_observations",
        "median_spearman_rho",
        "median_change_pct_per_10deg",
        "bootstrap_ci_low",
        "bootstrap_ci_high",
        "geometry_sensitive_screen",
    ],
]

baseline_vza_distribution = (
    vza_distribution_summary.loc[
        vza_distribution_summary["phase"] == "Baseline",
        [
            "class_code",
            "vza_p10",
            "vza_median",
            "vza_p90",
            "median_valid_pct",
        ],
    ]
)

baseline_daily_association = (
    daily_association.loc[
        daily_association["phase"] == "Baseline",
        [
            "class_code",
            "product",
            "dates",
            "spearman_rho",
            "vza_time_spearman",
            "linear_change_pp_per_10deg",
        ],
    ]
    .rename(
        columns={
            "dates": "daily_dates",
            "spearman_rho": "daily_spearman_rho",
            "linear_change_pp_per_10deg": (
                "daily_change_pp_per_10deg"
            ),
        }
    )
)

vza_decision_table = (
    baseline_effects
    .merge(
        baseline_vza_distribution,
        on="class_code",
        how="left",
    )
    .merge(
        baseline_daily_association,
        on=["class_code", "product"],
        how="left",
    )
    .sort_values(["class_code", "product"])
    .reset_index(drop=True)
)

vza_decision_table.to_csv(
    TABLE_DIR / "vza_reliability_decision_table.csv",
    index=False,
)

display(
    vza_decision_table.style.format(
        {
            "median_observations": "{:.0f}",
            "median_spearman_rho": "{:.3f}",
            "daily_spearman_rho": "{:.3f}",
            "vza_time_spearman": "{:.3f}",
            "daily_change_pp_per_10deg": "{:+.2f}",
            "median_change_pct_per_10deg": "{:+.2f}",
            "bootstrap_ci_low": "{:+.2f}",
            "bootstrap_ci_high": "{:+.2f}",
            "vza_p10": "{:.2f}",
            "vza_median": "{:.2f}",
            "vza_p90": "{:.2f}",
            "median_valid_pct": "{:.1f}",
        },
        na_rep="—",
    )
)

print("Tables:", TABLE_DIR)
print("Figures:", FIGURE_DIR)

,class_code,settlement_class,product,pixels,median_observations,median_spearman_rho,median_change_pct_per_10deg,bootstrap_ci_low,bootstrap_ci_high,geometry_sensitive_screen,vza_p10,vza_median,vza_p90,median_valid_pct,daily_dates,daily_spearman_rho,vza_time_spearman,daily_change_pp_per_10deg
0,11,Very low-density rural,DNB-BRDF,114,40,0.211,+4.99,+4.11,+5.88,False,9.31,44.11,63.44,67.5,64,0.339,-0.061,+9.12
1,11,Very low-density rural,Gap-filled operational,114,180,0.057,+0.95,+0.86,+1.14,False,9.31,44.11,63.44,67.5,64,0.358,-0.061,+4.93
2,11,Very low-density rural,Gap-filled paired,114,40,0.211,+4.99,+4.11,+5.88,False,9.31,44.11,63.44,67.5,64,0.339,-0.061,+9.12
3,12,Low-density rural,DNB-BRDF,150,37,0.248,+4.45,+3.99,+6.01,False,17.90,41.72,63.58,43.6,79,0.475,-0.060,+11.23
4,12,Low-density rural,Gap-filled operational,150,180,0.063,+0.95,+0.74,+1.13,False,17.90,41.72,63.58,43.6,79,0.267,-0.060,+4.10
5,12,Low-density rural,Gap-filled paired,150,37,0.248,+4.45,+3.99,+6.01,False,17.90,41.72,63.58,43.6,79,0.475,-0.060,+11.23
6,13,Rural cluster,DNB-BRDF,123,38,0.193,+4.14,+3.58,+4.88,False,18.96,41.80,63.70,52.0,73,0.399,-0.015,+8.41
7,13,Rural cluster,Gap-filled operational,123,180,0.050,+0.79,+0.66,+0.90,False,18.96,41.80,63.70,52.0,73,0.293,-0.015,+4.07
8,13,Rural cluster,Gap-filled paired,123,38,0.193,+4.14,+3.58,+4.88,False,18.96,41.80,63.70,52.0,73,0.399,-0.015,+8.41
9,21,Suburban / peri-urban,DNB-BRDF,150,34,0.289,+4.65,+4.21,+5.24,False,17.75,42.87,63.62,36.1,90,0.478,-0.013,+11.38


Tables: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/outputs/haiyan_vza_effects/g7_mqf_0/tables
Figures: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/outputs/haiyan_vza_effects/g7_mqf_0/figures


**Table 22 — Atomic-class VZA evidence.** This table identifies which settlement classes and products show coherent baseline angular sensitivity.

**RQ2 relevance:** a class-level positive screen is a warning, not an automatic correction mandate. It must be reconciled with threshold stability, response heterogeneity, correction factors, and retained observability.

In [44]:
final_vza_decision = vza_stability_decision.copy()

if periodicity_comparison.empty:
    final_vza_decision["orbit_r2_reduction_pct"] = np.nan
else:
    final_vza_decision = final_vza_decision.merge(
        periodicity_comparison[
            ["product", "orbit_r2_reduction_pct"]
        ],
        on="product",
        how="left",
    )

final_vza_decision["recommended_rq2_treatment"] = np.select(
    [
        final_vza_decision["decision"] == "Observation-limited",
        final_vza_decision["decision"] == "Geometry-sensitive",
    ],
    [
        "Do not estimate a unique recovery metric from this subset",
        "Report all-angle and thresholded sensitivity estimates",
    ],
    default="Retain all selected-quality angles; report VZA diagnosis",
)

final_vza_decision.to_csv(
    TABLE_DIR / f"{OUTPUT_TAG}_final_vza_decision.csv",
    index=False,
)

display(final_vza_decision.round(2))

,product,threshold_label,minimum_phase_dates,maximum_absolute_difference_pp,decision,orbit_r2_reduction_pct,recommended_rq2_treatment
0,DNB-BRDF,VZA ≤ 30°,11,7.99,Geometry-sensitive,50.23,Report all-angle and thresholded sensitivity e...
1,DNB-BRDF,VZA ≤ 40°,15,7.80,Geometry-sensitive,50.23,Report all-angle and thresholded sensitivity e...
2,Gap-filled paired,VZA ≤ 30°,11,7.99,Geometry-sensitive,50.23,Report all-angle and thresholded sensitivity e...
3,Gap-filled paired,VZA ≤ 40°,15,7.80,Geometry-sensitive,50.23,Report all-angle and thresholded sensitivity e...


**Table 23 — Final VZA decision for RQ2.** The decision integrates threshold stability, minimum phase support, and—when available—the reduction in orbit-structured variability after correction.

Interpret the classifications as follows:

- **Geometry-robust:** the main phase estimates remain stable; retain all selected-quality VZA and report the diagnostic.
- **Geometry-sensitive:** impact or recovery changes materially; report all-angle and thresholded estimates rather than one value.
- **Observation-limited:** stricter control leaves insufficient support; distinguish “not observable” from “not recovered.”

The central RQ2 result is therefore whether the Haiyan shock and recovery conclusions survive credible changes in VZA treatment while preserving observability.

## 12. Interpretation checklist

1. **Confirm observability.** Treat radiance changes on low-coverage days as uncertain because spatial support may have changed.
2. **Use the baseline for VZA inference.** Do not learn an angular response from shock or recovery observations.
3. **Prioritise within-pixel evidence.** It most directly tests whether the same locations brighten or dim with VZA.
4. **Keep direct, paired gap-filled, and operational gap-filled supports separate.**
5. **Test the 16-day orbit cycle.** Repeating radiance variation is not automatically recovery dynamics.
6. **Prefer threshold stability before model-based correction.**
7. **Retain correction as a sensitivity product** unless it has supported fits, plausible factors, reduced periodicity, and stable phase interpretation.
8. **Report the result as geometry-robust, geometry-sensitive, or observation-limited.**

**RQ2 handoff:** apply the selected VZA treatment to shock magnitude, minimum timing, T50/T80/T90, recovery slope, cumulative deficit, and post-event stability. Every metric should retain its MQF definition, GHSL group, VZA treatment, valid-pixel support, and uncertainty flag.